# CartPole, do it yourself: from Python and numpy basics to Q-learning and beyond

This notebook is built so that you **type everything**. The code is written in the markdown cells (the ones with text), fully commented. Below each one there is an empty code cell where *you* type the code, run it, and compare your result with the expected output in the next cell. Typing is slow on purpose: it is how the code ends up in your head rather than just on your screen.

**What you will build:** a physics simulation of a pole balanced on a cart, then an agent that *teaches itself* to keep the pole up, using Q-learning. Along the way you will learn the Python and numpy you need (no data-science background assumed), and at the end you will see two very different ways to solve the same problem, including one that needs only four numbers.

**The problem in one paragraph.** A pole is hinged on a cart that rolls along a track. Every 0.02 seconds you may push the cart left or right (only two choices). If the pole tilts more than 12 degrees, or the cart runs off the track (2.4 m from the centre), you fail. You earn +1 point for each tick you survive, and surviving 500 ticks counts as a win. The agent is never told how to balance. It only sees four numbers (where the cart is, how fast it moves, the pole's angle, how fast the angle changes) and the points it earns.

## How to use this notebook

**The loop for every step:**
1. Read the explanation above the code.
2. Type the code into the empty code cell below it. Type the comments too; they are short explanations you will be glad of later.
3. Run the cell (`Shift+Enter`).
4. Compare your output with the "Expected output" cell. Then read what the output means.
5. Do the "Try this yourself" bit when there is one, and predict the answer *before* running.

**Setup.** You need Python 3.9 or newer, and the packages `numpy`, `matplotlib` (optional) and `gymnasium` (only in the last step). In a terminal: `pip install numpy matplotlib gymnasium`. Use Jupyter or VS Code with the Jupyter extension. In VS Code, the `+ Code` button under a cell adds a code cell.

**Cells share memory.** A name you create in one cell (like `step` or `Q`) exists in later cells, as long as the kernel is running. Run the steps **in order**. If something behaves strangely, use *Restart* then run everything from the top.

**If your output does not match,** check, in this order: (1) a typo, especially brackets and commas; (2) indentation (Python uses the spaces at the start of a line to mean "this belongs inside the thing above"); (3) you skipped or reordered a step; (4) you left out a `random.seed(...)` line. Cells with randomness begin with a seed, which makes the "random" numbers repeatable, so your numbers should match exactly. Tiny differences in the last decimal places (the 15th or 16th digit) are harmless.

**Errors are normal.** Red text with `NameError` usually means you did not run an earlier step. `SyntaxError` means a typo. Read the last line of the error first: it says what went wrong and the arrow points at where.

## Read this first: how a computer reads code, and how to read the line-by-line sections

If you are new to programming, the most useful thing you can learn is to read code **the way the computer does**: one line at a time, working out the *value* and the *type* of everything. The sections called "Line by line" below do this for you with real numbers. Here is the vocabulary they use.

**A variable is a labelled box.** The line `x = 5` makes a box, sticks the label `x` on it, and puts the value `5` inside. Later lines use the label to get the value out. A new `x = 7` replaces what is in the box. The `=` sign means *store*, never "is equal to". (To ask "is it equal?" you write `==`.)

**Every value has a type**, which decides what you can do with it. The types in this notebook:

| Type | What it is | Example | Notes |
|---|---|---|---|
| `int` | whole number | `3` | `3 + 4` is `7` |
| `float` | decimal number | `0.05` | `3.0` is a float even though it looks whole |
| `bool` | True or False | `True` | the result of questions like `3 > 2` |
| `str` | text | `"cart"` | in quotes |
| `list` | ordered row of items, **changeable** | `[1, 2, 3]` | square brackets; items counted from 0 |
| `tuple` | ordered group, **fixed** once made | `(3, 4)` | round brackets; good for positions |
| `dict` | look-up table of key to value | `{(0, 0): 5}` | curly brackets |
| numpy array | block of numbers of one type, with fast maths | `np.array([1, 2])` | `*` and `+` act on every item |
| `function` | a stored recipe | `step` | runs only when *called* |

**The same symbol can mean different things for different types.** `[1, 2] * 2` repeats a list, giving `[1, 2, 1, 2]`. But `np.array([1, 2]) * 2` multiplies every item, giving `[2 4]`. So always ask: *what type is this?*

**Expressions are worked out from the inside outward.** Take `round(2 * (3 + 4), 1)`. The computer does the innermost brackets first: `3 + 4` becomes `7`; then `2 * 7` becomes `14`; then `round(14, 1)` becomes `14`. The sections below show this as a numbered list for every line, with the type and value of each piece, so you can check your own reasoning against it.

**A function is a recipe.** *Defining* it (`def step(state, action):`) only stores the recipe. *Calling* it (`step(my_state, 1)`) runs it: the values you pass in (the **arguments**) fill the empty slots (the **parameters**), the indented lines run top to bottom, and whatever follows `return` comes back out as the value of the call. A function name *without* brackets (`step`) is the recipe itself, which can be passed to other functions. A name *with* brackets (`step(...)`) means "run it now".

**Counting starts at 0.** In `[10, 20, 30]`, position 0 holds `10`, position 1 holds `20`. `x[1:3]` is a *slice*: positions 1 up to, but not including, 3.

**Indentation is meaning.** In Python the spaces at the start of a line show which lines belong inside a function, loop or `if`. Four spaces per level.

**Loops repeat lines.** `for k in range(3):` runs its indented lines three times, with `k` equal to 0, then 1, then 2. `while condition:` repeats as long as the condition is `True`. When a line sits in a loop, the notebook shows its **first pass** and tells you how many times it ran.

### How the "Line by line" sections are laid out

For every line you type you will see:

- **Creates:** the name, its **type** and its **value** at that moment. This answers "what *is* this thing?"
- **Evaluated inside-out:** the pieces of the expression in the order the computer works them out, each with its type and value.
- **Prints:** exactly what appears on screen.
- **Why / what it means:** why the line is there, in plain words.
- For **functions**, a trace of **one concrete call**: what goes in, what each line produces, and what comes back out.

You do not need to memorise the vocabulary. Whenever a word is new, it is explained where it first appears, and the table above is always here to look back at.

## The map of this notebook

- **Part A: the toolkit.** Python and numpy basics, each one introduced because the project needs it. (Steps 1 to 6)
- **Part B: build the world.** The physics of the cart and pole, written as code, plus a baseline of "dumb" players to beat. (Steps 7 to 12)
- **Part C: Q-learning.** What "reward", "return" and "Q" mean, then the table, the update rule and the training loop. (Steps 13 to 20)
- **Part D: experiments.** Break the learner on purpose to find out why each piece matters. (Steps 21 to 25)
- **Part E: beyond.** A completely different method that solves CartPole with four numbers, a check against the official version of the problem, and a map of the wider field. (Steps 26 to 28)

---
# Part A: the toolkit (Python and numpy)

You do not need to be a data scientist for this. Reinforcement learning leans on a small slice of numpy, and we will learn exactly that slice. Every idea below is used later in the project, and I will point out where.

## Step 1: Lists versus numpy arrays

**The idea first.** Python's built-in **list** holds anything in a row. A **numpy array** holds numbers (all the same type) in a compact block, and lets you do arithmetic on the whole block at once. That is called *vectorised* arithmetic. We will store the Q-table as a numpy array, and compute with whole arrays instead of writing loops.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 1.1: type this

```python
import numpy as np                        # load numpy and give it the nickname np
```

- **Creates `np`** — type: **module (a library of tools)**; value: `<module numpy>`

In [ ]:
# Step 1.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.2: type this

```python
a_list = [1, 2, 3]                        # an ordinary Python list
```

- **Creates `a_list`** — type: **list of 3 items (each an int)**; value: `[1, 2, 3]`

- **Why / what it means:** Square brackets build a **list**: an ordered row of items that you can add to, remove from and change. This one holds three `int`s (whole numbers). A list is Python's general-purpose container and can hold anything, even a mixture of types. The name `a_list` is a label we stick on the list so we can refer to it later.

In [ ]:
# Step 1.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.3: type this

```python
an_array = np.array([1, 2, 3])            # a numpy array built from that list
```

- **Creates `an_array`** — type: **numpy array, shape (3,), holding whole numbers**; value: `[1, 2, 3]`

- **Why / what it means:** `np.array(...)` is a **function call**: a function's name, then parentheses holding the *argument* we hand to it. We hand it the list, and it returns a different kind of object, a numpy **array**. The digits are the same but the object is not: an array is one tight block of numbers of a single type, and it knows how to do arithmetic on the whole block at once. A list does not. That difference is why the next four lines behave differently.

In [ ]:
# Step 1.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.4: type this

```python
print(a_list * 2)                         # on a LIST, * means "repeat the list"
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `a_list` → *list of 3 items (each an int)*: `[1, 2, 3]`
2. `a_list * 2` → *list of 6 items (each an int)*: `[1, 2, 3, 1, 2, 3]`

**Prints:**
```text
[1, 2, 3, 1, 2, 3]
```

In [ ]:
# Step 1.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.5: type this

```python
print(an_array * 2)                       # on an ARRAY, * means "multiply every item"
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `an_array` → *numpy array, shape (3,), holding whole numbers*: `[1, 2, 3]`
2. `an_array * 2` → *numpy array, shape (3,), holding whole numbers*: `[2, 4, 6]`

**Prints:**
```text
[2 4 6]
```

In [ ]:
# Step 1.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.6: type this

```python
print(an_array + 10)                      # every item gets 10 added
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `an_array` → *numpy array, shape (3,), holding whole numbers*: `[1, 2, 3]`
2. `an_array + 10` → *numpy array, shape (3,), holding whole numbers*: `[11, 12, 13]`

**Prints:**
```text
[11 12 13]
```

In [ ]:
# Step 1.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.7: type this

```python
print(an_array + np.array([10, 20, 30]))  # arrays of equal size add item by item
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `an_array` → *numpy array, shape (3,), holding whole numbers*: `[1, 2, 3]`
2. `np.array([10, 20, 30])` → *numpy array, shape (3,), holding whole numbers*: `[10, 20, 30]`
3. `an_array + np.array([10, 20, 30])` → *numpy array, shape (3,), holding whole numbers*: `[11, 22, 33]`

**Prints:**
```text
[11 22 33]
```

In [ ]:
# Step 1.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.8: type this

```python
print(an_array.shape, an_array.size)      # shape = size in each dimension; size = how many items
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `an_array` → *numpy array, shape (3,), holding whole numbers*: `[1, 2, 3]`
2. `an_array.shape` → *tuple of 1 item (each an int)*: `(3,)`
3. `an_array.size` → *int (whole number)*: `3`

**Prints:**
```text
(3,) 3
```

In [ ]:
# Step 1.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1 complete: everything it printed, and what it means

```text
[1, 2, 3, 1, 2, 3]
[2 4 6]
[11 12 13]
[11 22 33]
(3,) 3
```

The list `[1, 2, 3, 1, 2, 3]` shows that `*` on a list *repeats* it. The array results `[2 4 6]`, `[11 12 13]` and `[11 22 33]` show arithmetic applied item by item. Note that arrays print **without commas**, which is how you tell an array from a list at a glance. `(3,)` is the shape of a one-dimensional array of three items (the odd comma is how Python writes a tuple with one element), and `3` is its size.

**Try this yourself.** Predict, then check: what do `an_array * an_array` and `an_array ** 2` give? (`**` means "to the power of".)

## Step 2: Two-dimensional arrays and indexing

**The idea first.** A table with rows and columns is a 2D array. Our Q-table will have one row per situation and one column per action, and we will constantly need to say "give me this row" or "this one box".

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 2.1: type this

```python
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])              # a 2D array: 2 rows, 3 columns
```

- **Creates `grid`** — type: **numpy array, shape (2, 3), holding whole numbers**; value: `[[1, 2, 3], [4, 5, 6]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[1, 2, 3], [4, 5, 6]]` → *list of 2 items (each a list)*: `[[1, 2, 3], [4, 5, 6]]`

- **Why / what it means:** This is a list *inside* a list: each inner list `[1, 2, 3]` and `[4, 5, 6]` becomes one **row**. `np.array` turns the nested lists into a 2D array whose shape is `(2, 3)`: **2 rows, 3 columns** (rows are always listed first). The shape is itself a tuple: a fixed group of numbers.

In [ ]:
# Step 2.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.2: type this

```python
print(grid.shape)                         # (rows, columns)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid.shape` → *tuple of 2 items (each an int)*: `(2, 3)`

**Prints:**
```text
(2, 3)
```

In [ ]:
# Step 2.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.3: type this

```python
print(grid[0, 2])                         # row 0, column 2 (counting starts at 0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid[0, 2]` → *numpy integer (a whole number)*: `3`

**Prints:**
```text
3
```

In [ ]:
# Step 2.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.4: type this

```python
print(grid[1])                            # the whole of row 1
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid[1]` → *numpy array, shape (3,), holding whole numbers*: `[4, 5, 6]`

**Prints:**
```text
[4 5 6]
```

In [ ]:
# Step 2.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.5: type this

```python
print(grid[:, 0])                         # ":" means "all rows", so this is column 0
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid[:, 0]` → *numpy array, shape (2,), holding whole numbers*: `[1, 4]`

**Prints:**
```text
[1 4]
```

In [ ]:
# Step 2.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.6: type this

```python
print(grid[0, 1:])                        # row 0, columns from 1 to the end
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid[0, 1:]` → *numpy array, shape (2,), holding whole numbers*: `[2, 3]`

**Prints:**
```text
[2 3]
```

In [ ]:
# Step 2.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.7: type this

```python
print(grid.reshape(3, 2))                 # same six numbers, rearranged to 3 rows x 2 columns
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid.reshape(3, 2)` → *numpy array, shape (3, 2), holding whole numbers*: `[[1, 2], [3, 4], [5, 6]]`

**Prints:**
```text
[[1 2]
 [3 4]
 [5 6]]
```

In [ ]:
# Step 2.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.8: type this

```python
print(grid[(1, 2)])                       # a tuple (1, 2) works as the index too
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `grid` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 2, 3], [4, 5, 6]]`
2. `grid[1, 2]` → *numpy integer (a whole number)*: `6`

**Prints:**
```text
6
```

In [ ]:
# Step 2.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2 complete: everything it printed, and what it means

```text
(2, 3)
3
[4 5 6]
[1 4]
[2 3]
[[1 2]
 [3 4]
 [5 6]]
6
```

`(2, 3)` is rows by columns. `3` is the item at row 0, column 2. `[4 5 6]` is row 1. `[1 4]` is column 0 (first item of each row). `[2 3]` is row 0 from column 1 on. The reshaped array is the same numbers refilled across three rows. The last `6` is `grid[1, 2]`, the bottom-right item.

**Try this yourself.** Predict, then check: `grid[1, :2]` (the first two items of row 1) and `grid[-1]` (a negative index counts from the end).

## Step 3: Axes: max, argmax, sum and zeros

**The idea first.** Two operations sit at the centre of Q-learning: **max** (the best number in a row) and **argmax** (the *position* of the best number, which tells you which action is best). To use them on a 2D table you must understand `axis`.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 3.1: type this

```python
scores = np.array([[3, 9, 1],
                   [7, 2, 8]])
```

- **Creates `scores`** — type: **numpy array, shape (2, 3), holding whole numbers**; value: `[[3, 9, 1], [7, 2, 8]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[3, 9, 1], [7, 2, 8]]` → *list of 2 items (each a list)*: `[[3, 9, 1], [7, 2, 8]]`

- **Why / what it means:** A 2 x 3 array of whole numbers. We chose numbers where the largest in each row is in a different column, so `argmax` has something interesting to tell us.

In [ ]:
# Step 3.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.2: type this

```python
print(scores.max())                       # biggest number anywhere
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *numpy array, shape (2, 3), holding whole numbers*: `[[3, 9, 1], [7, 2, 8]]`
2. `scores.max()` → *numpy integer (a whole number)*: `9`

**Prints:**
```text
9
```

In [ ]:
# Step 3.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.3: type this

```python
print(scores.max(axis=1))                 # axis=1: squash the columns -> one answer per ROW
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *numpy array, shape (2, 3), holding whole numbers*: `[[3, 9, 1], [7, 2, 8]]`
2. `scores.max(axis=1)` → *numpy array, shape (2,), holding whole numbers*: `[9, 8]`

**Prints:**
```text
[9 8]
```

In [ ]:
# Step 3.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.4: type this

```python
print(scores.max(axis=0))                 # axis=0: squash the rows -> one answer per COLUMN
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *numpy array, shape (2, 3), holding whole numbers*: `[[3, 9, 1], [7, 2, 8]]`
2. `scores.max(axis=0)` → *numpy array, shape (3,), holding whole numbers*: `[7, 9, 8]`

**Prints:**
```text
[7 9 8]
```

In [ ]:
# Step 3.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.5: type this

```python
print(scores.argmax(axis=1))              # POSITION of the biggest number in each row
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *numpy array, shape (2, 3), holding whole numbers*: `[[3, 9, 1], [7, 2, 8]]`
2. `scores.argmax(axis=1)` → *numpy array, shape (2,), holding whole numbers*: `[1, 2]`

**Prints:**
```text
[1 2]
```

In [ ]:
# Step 3.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.6: type this

```python
print(scores.sum(), scores.mean())        # total and average
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *numpy array, shape (2, 3), holding whole numbers*: `[[3, 9, 1], [7, 2, 8]]`
2. `scores.sum()` → *numpy integer (a whole number)*: `30`
3. `scores.mean()` → *float (decimal number)*: `5.0`

**Prints:**
```text
30 5.0
```

In [ ]:
# Step 3.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.7: type this

```python
print(np.zeros((2, 3)))                   # a 2x3 array of zeros
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.zeros((2, 3))` → *numpy array, shape (2, 3), holding decimal numbers*: `[[0., 0., 0.], [0., 0., 0.]]`

**Prints:**
```text
[[0. 0. 0.]
 [0. 0. 0.]]
```

In [ ]:
# Step 3.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.8: type this

```python
print(np.zeros((2, 2, 2)).shape)          # arrays can have 3, 4, 5... dimensions
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.zeros((2, 2, 2))` → *numpy array, shape (2, 2, 2), holding decimal numbers*: `[[[0., 0.], [0., 0.]], [[0., 0.], [0., 0.]]]`
2. `np.zeros((2, 2, 2)).shape` → *tuple of 3 items (each an int)*: `(2, 2, 2)`

**Prints:**
```text
(2, 2, 2)
```

In [ ]:
# Step 3.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3 complete: everything it printed, and what it means

```text
9
[9 8]
[7 9 8]
[1 2]
30 5.0
[[0. 0. 0.]
 [0. 0. 0.]]
(2, 2, 2)
```

`9`; then `[9 8]` (best per row); `[7 9 8]` (best per column); `[1 2]` (positions of the best in each row); `30 5.0` (sum and mean). Then a 2x3 block of zeros, and the shape `(2, 2, 2)`. In Q-learning, `Q[s].max()` answers "how good is this situation?", and `Q[s].argmax()` answers "which action should I take here?".

**Try this yourself.** Predict, then check: what are `scores.sum(axis=0)` and `scores.mean(axis=1)`?

## Step 4: Comparisons, masks, clip and astype

**The idea first.** We need to ask questions of a whole array at once ("which numbers are bigger than 1?"), force numbers into a range, and convert decimals to whole numbers. The discretizing function and the tie-breaking code later both use these.

This step has **6 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 4.1: type this

```python
v = np.array([0.2, 1.7, -0.4, 2.9])
```

- **Creates `v`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.2, 1.7, -0.4, 2.9]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[0.2, 1.7, -0.4, 2.9]` → *list of 4 items (each a float)*: `[0.2, 1.7, -0.4, 2.9]`

- **Why / what it means:** A one-dimensional array of four `float`s (decimal numbers). Because we wrote decimal points, numpy stores them as decimals rather than whole numbers. This array is only a playground for the comparison tools that follow.

In [ ]:
# Step 4.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.2: type this

```python
print(v > 1)                              # compares every item: gives True/False for each
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `v` → *numpy array, shape (4,), holding decimal numbers*: `[0.2, 1.7, -0.4, 2.9]`
2. `v > 1` → *numpy array, shape (4,), holding True/False values*: `[False, True, False, True]`

**Prints:**
```text
[False  True False  True]
```

In [ ]:
# Step 4.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.3: type this

```python
print(np.flatnonzero(v > 1))              # the POSITIONS where the answer is True
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `v` → *numpy array, shape (4,), holding decimal numbers*: `[0.2, 1.7, -0.4, 2.9]`
2. `v > 1` → *numpy array, shape (4,), holding True/False values*: `[False, True, False, True]`
3. `np.flatnonzero(v > 1)` → *numpy array, shape (2,), holding whole numbers*: `[1, 3]`

**Prints:**
```text
[1 3]
```

In [ ]:
# Step 4.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.4: type this

```python
print(np.clip(v, 0, 2))                   # force every item into the range 0 to 2
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `v` → *numpy array, shape (4,), holding decimal numbers*: `[0.2, 1.7, -0.4, 2.9]`
2. `np.clip(v, 0, 2)` → *numpy array, shape (4,), holding decimal numbers*: `[0.2, 1.7, 0., 2.]`

**Prints:**
```text
[0.2 1.7 0.  2. ]
```

In [ ]:
# Step 4.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.5: type this

```python
print(v.astype(int))                      # convert to whole numbers by chopping off decimals
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `v` → *numpy array, shape (4,), holding decimal numbers*: `[0.2, 1.7, -0.4, 2.9]`
2. `v.astype(int)` → *numpy array, shape (4,), holding whole numbers*: `[0, 1, 0, 2]`

**Prints:**
```text
[0 1 0 2]
```

In [ ]:
# Step 4.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.6: type this

```python
print(np.array([1, 5, 5, 2]) == 5)        # == tests equality item by item
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.array([1, 5, 5, 2])` → *numpy array, shape (4,), holding whole numbers*: `[1, 5, 5, 2]`
2. `np.array([1, 5, 5, 2]) == 5` → *numpy array, shape (4,), holding True/False values*: `[False, True, True, False]`

**Prints:**
```text
[False  True  True False]
```

In [ ]:
# Step 4.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4 complete: everything it printed, and what it means

```text
[False  True False  True]
[1 3]
[0.2 1.7 0.  2. ]
[0 1 0 2]
[False  True  True False]
```

`[False True False True]` is the mask: items 1 and 3 exceed 1. `[1 3]` are those positions. The clipped array is `[0.2 1.7 0. 2.]` (the -0.4 became 0, the 2.9 became 2). The `astype(int)` result is `[0 1 0 2]`. The last line shows which items equal 5: positions 1 and 2.

**Try this yourself.** Predict: what does `np.clip(np.array([5, -3, 8]), 0, 6)` give?

## Step 5: Random numbers and seeds

**The idea first.** Learning agents need randomness: random starting positions, and random exploring moves. But a computer cannot be truly random; it follows a formula that *looks* random. The starting point of that formula is the **seed**. Same seed, same sequence. We use seeds so that your results match mine and experiments can be repeated.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 5.1: type this

```python
import random                             # Python's built-in random number tools
```

- **Creates `random`** — type: **module (a library of tools)**; value: `<module random>`

In [ ]:
# Step 5.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.2: type this

```python
random.seed(42)                           # fix the starting point of the random sequence
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(42)` → *None (nothing)*: `None`

- **Why / what it means:** `random.seed(42)` does not produce a number. It sets the *starting point* of Python's random-number formula to 42. The formula is deterministic, so from this starting point the sequence of 'random' numbers that follows is always the same. This is what lets your numbers match mine.

In [ ]:
# Step 5.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.3: type this

```python
print(random.random())                    # a decimal between 0 and 1
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.random()` → *float (decimal number)*: `0.6394267984578837`

**Prints:**
```text
0.6394267984578837
```

In [ ]:
# Step 5.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.4: type this

```python
print(random.uniform(-0.05, 0.05))        # a decimal between two limits
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.uniform(-0.05, 0.05)` → *float (decimal number)*: `-0.04749892447773331`

**Prints:**
```text
-0.04749892447773331
```

In [ ]:
# Step 5.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.5: type this

```python
print(random.randrange(2))                # a whole number: 0 or 1
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.randrange(2)` → *int (whole number)*: `1`

**Prints:**
```text
1
```

In [ ]:
# Step 5.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.6: type this

```python
print(random.choice([10, 20, 30]))        # one item picked from a list
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.choice([10, 20, 30])` → *int (whole number)*: `10`

**Prints:**
```text
10
```

In [ ]:
# Step 5.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.7: type this

```python
random.seed(42)                           # restart from the same point...
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(42)` → *None (nothing)*: `None`

- **Why / what it means:** Setting the same seed again rewinds the formula to the same starting point, so the next `random.random()` repeats the very first number you saw.

In [ ]:
# Step 5.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.8: type this

```python
print(random.random())                    # ...and we get the SAME first number again
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.random()` → *float (decimal number)*: `0.6394267984578837`

**Prints:**
```text
0.6394267984578837
```

In [ ]:
# Step 5.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5 complete: everything it printed, and what it means

```text
0.6394267984578837
-0.04749892447773331
1
10
0.6394267984578837
```

The first and last numbers are the same, `0.6394267984578837`, because the seed was reset. The other three values depend on the seed and on Python's generator. If you are on a very different Python version and one of those differs, do not worry about this step; later steps use the same functions, so if *they* match, you are fine.

**Try this yourself.** Remove the second `random.seed(42)` line. Does the final number still repeat the first? Why not?

## Step 6: Functions, tuples, comprehensions, f-strings and loops

**The idea first.** Five small Python tools that the whole project uses. If any are already familiar, skim the explanation and still type the cell.

This step has **11 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 6.1: type this

```python
def add(x, y):                            # def starts a function; x and y are its inputs
    return x + y                          # return hands a result back to the caller
```

- **Creates** `add` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** `def` *defines* a function: it stores a recipe under the name `add`. Nothing runs yet. `x` and `y` are **parameters**: empty slots that receive values when someone calls the function. The indented line is the **body**, and `return` hands a result back to whoever called it. Indentation (the four spaces) is how Python knows which lines belong to the function.

In [ ]:
# Step 6.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.2: type this

```python
print(add(2, 3))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `add(2, 3)` → *int (whole number)*: `5`

**Prints:**
```text
5
```

In [ ]:
# Step 6.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.3: type this

```python
point = (3, 4)                            # a tuple: a fixed group of values
```

- **Creates `point`** — type: **tuple of 2 items (each an int)**; value: `(3, 4)`

- **Why / what it means:** Round brackets build a **tuple**: a fixed group of values, like a list that cannot be changed after it is made. Tuples are ideal for things that belong together permanently, like a (row, column) position. That is exactly how we will store cell positions in the maze and addresses in the Q-table.

In [ ]:
# Step 6.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.4: type this

```python
row, col = point                          # "unpacking": row gets 3, col gets 4
```

- **Creates `row`** — type: **int (whole number)**; value: `3`

- **Creates `col`** — type: **int (whole number)**; value: `4`

- **Why / what it means:** **Unpacking:** a tuple of two items assigned to two names hands the first item to `row` and the second to `col`. The names on the left are created at the same moment.

In [ ]:
# Step 6.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.5: type this

```python
print(row, col)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `row` → *int (whole number)*: `3`
2. `col` → *int (whole number)*: `4`

**Prints:**
```text
3 4
```

In [ ]:
# Step 6.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.6: type this

```python
squares = [n * n for n in range(5)]       # list comprehension: build a list in one line
```

- **Creates `squares`** — type: **list of 5 items (each an int)**; value: `[0, 1, 4, 9, 16]`

- **Why / what it means:** A **list comprehension** is a compact loop that builds a list. Read it aloud: *'make a list of `n * n` for each `n` in `range(5)`'*. `range(5)` produces 0, 1, 2, 3, 4 (it stops *before* 5). Each `n` is squared and collected. The first pass has `n = 0`, giving 0; the last has `n = 4`, giving 16.

In [ ]:
# Step 6.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.7: type this

```python
print(squares)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `squares` → *list of 5 items (each an int)*: `[0, 1, 4, 9, 16]`

**Prints:**
```text
[0, 1, 4, 9, 16]
```

In [ ]:
# Step 6.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.8: type this

```python
print(f"row {row}, col {col}, sum {row + col:5.1f}")   # f-string: {value:width.decimals}
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `row` → *int (whole number)*: `3`
2. `col` → *int (whole number)*: `4`
3. `row + col` → *int (whole number)*: `7`

**Prints:**
```text
row 3, col 4, sum   7.0
```

In [ ]:
# Step 6.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.9: type this

```python
count = 0
```

- **Creates `count`** — type: **int (whole number)**; value: `0`

- **Why / what it means:** A starting value for a counter. We create it before the loop so the loop has something to add to.

In [ ]:
# Step 6.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.10: type this

```python
while count < 10:                         # repeat while the condition stays True
    count += 1                            # += means "add to the existing value"
    if count == 3:
        break                             # break leaves the loop immediately
```

- **Creates `count`** — type: **int (whole number)**; value: `3`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `count` → *int (whole number)*: `0` *(evaluated 3 times; the first is shown)*
2. `count < 10` → *bool (True/False)*: `True` *(evaluated 3 times; the first is shown)*
3. `count == 3` → *bool (True/False)*: `False` *(evaluated 3 times; the first is shown)*

- **Why / what it means:** The `while` loop repeats its indented lines for as long as `count < 10` is true. `count += 1` is shorthand for `count = count + 1`: read the current value, add 1, store it back. The `if` inside checks for 3 and `break` ends the loop immediately, which is why the final value is 3 and not 10.

In [ ]:
# Step 6.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.11: type this

```python
print(count)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `count` → *int (whole number)*: `3`

**Prints:**
```text
3
```

In [ ]:
# Step 6.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6 complete: everything it printed, and what it means

```text
5
3 4
[0, 1, 4, 9, 16]
row 3, col 4, sum   7.0
3
```

`5` is `add(2, 3)`. `3 4` shows the unpacked tuple. `[0, 1, 4, 9, 16]` is the comprehension. The f-string line prints `row 3, col 4, sum   7.0`, where the extra spaces come from the width of 5. The loop prints `3` because `break` fired when `count` reached 3.

**Try this yourself.** Write a comprehension that gives the list `[0, 2, 4, 6, 8]` (hint: multiply by 2 instead of squaring).

---
# Part B: build the world

Before an agent can learn to balance a pole, there has to be a pole. We write the physics ourselves. This is called an **environment**: the part of the system that, given a situation and a choice, tells you what happens next. The agent will live inside it.

## Step 7: The constants of the world

**The idea first.** The physics needs a handful of fixed numbers: how strong gravity is, how heavy the cart and pole are, how hard we push, and how long each tick lasts. We name them once at the top, in CAPITALS (the Python convention for "this never changes"), so the physics code reads clearly.

This step has **11 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 7.1: type this

```python
import math                               # sin, cos, pi
```

- **Creates `math`** — type: **module (a library of tools)**; value: `<module math>`

In [ ]:
# Step 7.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.2: type this

```python
GRAVITY = 9.8                             # m/s^2: how fast gravity accelerates things
MASS_CART = 1.0                           # kg
MASS_POLE = 0.1                           # kg
```

- **Creates `GRAVITY`** — type: **float (decimal number)**; value: `9.8`

- **Why / what it means:** A **constant**: a name for a number that never changes while the program runs (written in CAPITALS by convention). It is a `float`, because physical quantities are rarely whole numbers. Python does not store units; the unit (metres per second squared) lives only in the comment, so *you* must keep your units consistent.

- **Creates `MASS_CART`** — type: **float (decimal number)**; value: `1.0`

- **Why / what it means:** The cart's mass in kilograms, as a `float`. We write `1.0` rather than `1` so it is a decimal number; mixing whole numbers and decimals works in Python, but staying consistent avoids surprises.

- **Creates `MASS_POLE`** — type: **float (decimal number)**; value: `0.1`

- **Why / what it means:** The pole is light compared with the cart, which is part of what makes balancing possible.

In [ ]:
# Step 7.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.3: type this

```python
TOTAL_MASS = MASS_CART + MASS_POLE        # kg: the whole thing being pushed
```

- **Creates `TOTAL_MASS`** — type: **float (decimal number)**; value: `1.1`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `MASS_CART` → *float (decimal number)*: `1.0`
2. `MASS_POLE` → *float (decimal number)*: `0.1`

- **Why / what it means:** Our first **calculated** constant. Python reads the two names, finds their values (1.0 and 0.1), adds them, and stores the result under the new name. Notice it prints as `1.1`. It is needed because pushing the cart also moves the pole, so the force must accelerate the whole thing.

In [ ]:
# Step 7.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.4: type this

```python
HALF_POLE = 0.5                           # m: distance from hinge to the pole's centre
```

- **Creates `HALF_POLE`** — type: **float (decimal number)**; value: `0.5`

- **Why / what it means:** Physics equations use the distance from the hinge to the pole's centre of mass. For a uniform pole of length 1 m, that is half its length.

In [ ]:
# Step 7.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.5: type this

```python
POLE_MASS_LEN = MASS_POLE * HALF_POLE     # appears in the equations as one lump
```

- **Creates `POLE_MASS_LEN`** — type: **float (decimal number)**; value: `0.05`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `MASS_POLE` → *float (decimal number)*: `0.1`
2. `HALF_POLE` → *float (decimal number)*: `0.5`

- **Why / what it means:** Pole mass times that distance. The cart-pole equations contain this product several times, so we calculate it once and give it a name.

In [ ]:
# Step 7.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.6: type this

```python
FORCE = 10.0                              # newtons: strength of each push
TAU = 0.02                                # seconds per tick
X_LIMIT = 2.4                             # metres: the track edge
```

- **Creates `FORCE`** — type: **float (decimal number)**; value: `10.0`

- **Why / what it means:** How hard each push is, in newtons. We only have two choices, push with +10 or with -10, so this single number is the entire 'strength' of our control.

- **Creates `TAU`** — type: **float (decimal number)**; value: `0.02`

- **Why / what it means:** How much time passes in one tick. This is the **time step**. A smaller value would make the simulation more precise but slower; 0.02 s (50 ticks per second) is the standard choice for this problem.

- **Creates `X_LIMIT`** — type: **float (decimal number)**; value: `2.4`

- **Why / what it means:** The edge of the track. The cart fails if it goes beyond this distance from the centre in either direction.

In [ ]:
# Step 7.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.7: type this

```python
THETA_LIMIT = 12 * math.pi / 180          # 12 degrees converted to radians
```

- **Creates `THETA_LIMIT`** — type: **float (decimal number)**; value: `0.20943951023931953`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `math.pi` → *float (decimal number)*: `3.141592653589793`
2. `12 * math.pi` → *float (decimal number)*: `37.69911184307752`

- **Why / what it means:** A calculated constant, converting 12 degrees to **radians**. The expression is evaluated left to right: first `12 * math.pi` (a float), then divided by 180. `math.pi` is a number stored inside the `math` module. We need radians because `math.sin` and `math.cos` expect them.

In [ ]:
# Step 7.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.8: type this

```python
MAX_STEPS = 500                           # surviving this many ticks counts as a win
```

- **Creates `MAX_STEPS`** — type: **int (whole number)**; value: `500`

- **Why / what it means:** A whole number (`int`): the number of ticks that counts as winning. We use an `int` here because we will count ticks with whole numbers.

In [ ]:
# Step 7.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.9: type this

```python
print(TOTAL_MASS, POLE_MASS_LEN)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `TOTAL_MASS` → *float (decimal number)*: `1.1`
2. `POLE_MASS_LEN` → *float (decimal number)*: `0.05`

**Prints:**
```text
1.1 0.05
```

In [ ]:
# Step 7.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.10: type this

```python
print(THETA_LIMIT)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `THETA_LIMIT` → *float (decimal number)*: `0.20943951023931953`

**Prints:**
```text
0.20943951023931953
```

In [ ]:
# Step 7.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7.11: type this

```python
print(math.degrees(THETA_LIMIT))          # convert back to check
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `THETA_LIMIT` → *float (decimal number)*: `0.20943951023931953`
2. `math.degrees(THETA_LIMIT)` → *float (decimal number)*: `12.0`

**Prints:**
```text
12.0
```

In [ ]:
# Step 7.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7 complete: everything it printed, and what it means

```text
1.1 0.05
0.20943951023931953
12.0
```

`1.1 0.05` are the total mass and the combined lump. `0.2094...` is 12 degrees in radians, and `math.degrees` converts it back to `12.0`, confirming the conversion.

## Step 8: The state, and starting a new episode

**The idea first.** The **state** is everything needed to predict what happens next. For CartPole it is exactly four numbers. An **episode** is one attempt, from a fresh start until failure (or 500 ticks). `reset()` creates the starting state.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 8.1: type this

```python
state = [0.0, 0.0, 0.05, 0.0]             # [x, x_dot, theta, theta_dot]
```

- **Creates `state`** — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.05, 0.0]`

- **Why / what it means:** We store the state as a **list of four floats**. A list is a good choice because we will build new states constantly and want to read them by position. By convention the order is `[x, x_dot, theta, theta_dot]` (position, velocity, angle, spin), and every function in this notebook relies on that order.

In [ ]:
# Step 8.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.2: type this

```python
x, x_dot, theta, theta_dot = state        # unpack the list into four names
```

- **Creates `x`** — type: **float (decimal number)**; value: `0.0`

- **Creates `x_dot`** — type: **float (decimal number)**; value: `0.0`

- **Creates `theta`** — type: **float (decimal number)**; value: `0.05`

- **Creates `theta_dot`** — type: **float (decimal number)**; value: `0.0`

- **Why / what it means:** Unpacking a list: four names on the left, four items on the right, matched **by position**. After this line each name holds one plain `float`, which is easier to use in formulas than `state[2]`.

In [ ]:
# Step 8.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.3: type this

```python
print(x, x_dot, theta, theta_dot)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `x` → *float (decimal number)*: `0.0`
2. `x_dot` → *float (decimal number)*: `0.0`
3. `theta` → *float (decimal number)*: `0.05`
4. `theta_dot` → *float (decimal number)*: `0.0`

**Prints:**
```text
0.0 0.0 0.05 0.0
```

In [ ]:
# Step 8.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.4: type this

```python
def reset():
    # four small random numbers: a nearly balanced pole on a nearly still cart
    return [random.uniform(-0.05, 0.05) for _ in range(4)]
```

- **Creates** `reset` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A function with no parameters (empty brackets): it needs no input to make a new starting state. Because it uses `random`, calling it twice gives two different answers, unlike most functions.

In [ ]:
# Step 8.4: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `reset()`

**Now each line runs, top to bottom:**

```python
return [random.uniform(-0.05, 0.05) for _ in range(4)]
```
Evaluated inside-out (the main pieces):
1. `[random.uniform(-0.05, 0.05) for _ in range(4)]` → *list of 4 items (each a float)*: `[-0.04061404132257651, -0.04716525234779937, 0.033576510391986975, -0.006723293209494663]`
- **Now `_` holds** — type: **int (whole number)**; value: `3`
- **Why:** A list comprehension that repeats four times (`for _ in range(4)`; the `_` means 'I do not need the loop counter'). Each time it calls `random.uniform(-0.05, 0.05)`, which returns a decimal anywhere in that range with equal chance. So the function returns a **list of 4 floats**, all tiny: the pole and cart start almost perfectly balanced and almost still.

**The function hands back** (its *return value*) — type: **list of 4 items (each a float)**; value: `[-0.04061404132257651, -0.04716525234779937, 0.033576510391986975, -0.006723293209494663]`


### Step 8.5: type this

```python
random.seed(1)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(1)` → *None (nothing)*: `None`

- **Why / what it means:** Fixes the random sequence so that your two `reset()` calls give the same two lists as mine.

In [ ]:
# Step 8.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.6: type this

```python
print(reset())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `reset()` → *list of 4 items (each a float)*: `[-0.03656357558875988, 0.03474337369372327, 0.02637746189766141, -0.024493097426057833]`

**Prints:**
```text
[-0.03656357558875988, 0.03474337369372327, 0.02637746189766141, -0.024493097426057833]
```

In [ ]:
# Step 8.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.7: type this

```python
print(reset())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `reset()` → *list of 4 items (each a float)*: `[-0.0004564912908059035, -0.005050893521126185, 0.0151592972722763, 0.02887233511355132]`

**Prints:**
```text
[-0.0004564912908059035, -0.005050893521126185, 0.0151592972722763, 0.02887233511355132]
```

In [ ]:
# Step 8.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8 complete: everything it printed, and what it means

```text
0.0 0.0 0.05 0.0
[-0.03656357558875988, 0.03474337369372327, 0.02637746189766141, -0.024493097426057833]
[-0.0004564912908059035, -0.005050893521126185, 0.0151592972722763, 0.02887233511355132]
```

The first line echoes the four numbers. The next two lines are two different random start states: each call to `reset()` produces new numbers because the random generator advances. Every start is close to perfect balance, and it is the agent's job to keep it there.

## Step 9: The physics, one piece at a time

**The idea first.** This is the heart of the environment. Given the state and a push, it computes the state 0.02 seconds later. We do it *once, by hand, printing every intermediate number*, before wrapping it in a function. The equations are the standard cart-pole equations from the original 1983 paper by Barto, Sutton and Anderson, also used by the official Gym/Gymnasium CartPole. You do not need to derive them; you need to understand what each line **means**.

This step has **15 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 9.1: type this

```python
state = [0.0, 0.0, 0.05, 0.0]
action = 1                                # 1 = push right, 0 = push left
```

- **Creates `state`** — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.05, 0.0]`

- **Why / what it means:** A hand-picked starting state: motionless, with the pole leaning slightly right (0.05 radians is about 3 degrees). Using a chosen state rather than a random one lets us follow the numbers.

- **Creates `action`** — type: **int (whole number)**; value: `1`

- **Why / what it means:** An `int`. Our convention: 0 means push left, 1 means push right.

In [ ]:
# Step 9.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.2: type this

```python
x, x_dot, theta, theta_dot = state
```

- **Creates `x`** — type: **float (decimal number)**; value: `0.0`

- **Creates `x_dot`** — type: **float (decimal number)**; value: `0.0`

- **Creates `theta`** — type: **float (decimal number)**; value: `0.05`

- **Creates `theta_dot`** — type: **float (decimal number)**; value: `0.0`

- **Why / what it means:** Unpacking, as before: four separate `float`s, so the formulas below can use readable names.

In [ ]:
# Step 9.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.3: type this

```python
force = FORCE if action == 1 else -FORCE  # one-line if/else
```

- **Creates `force`** — type: **float (decimal number)**; value: `10.0`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `action` → *int (whole number)*: `1`
2. `action == 1` → *bool (True/False)*: `True`
3. `FORCE` → *float (decimal number)*: `10.0`

- **Why / what it means:** A **conditional expression**: *value-if-true* `if` *condition* `else` *value-if-false*. First the condition `action == 1` is evaluated to a `bool` (`True` here). Because it is `True`, Python picks the first value, `FORCE`. The minus sign in `-FORCE` flips the sign of a number, so a left push is -10.

In [ ]:
# Step 9.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.4: type this

```python
sin_t = math.sin(theta)
```

- **Creates `sin_t`** — type: **float (decimal number)**; value: `0.04997916927067833`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `theta` → *float (decimal number)*: `0.05`

- **Why / what it means:** `math.sin` takes an angle in radians and returns a number between -1 and 1. Gravity pulls straight down, but the pole can only swing about its hinge, so only the sideways part of gravity matters, and that is proportional to the sine of the angle. For this small angle the sine is almost the angle itself.

In [ ]:
# Step 9.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.5: type this

```python
cos_t = math.cos(theta)
```

- **Creates `cos_t`** — type: **float (decimal number)**; value: `0.9987502603949663`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `theta` → *float (decimal number)*: `0.05`

- **Why / what it means:** The cosine of a small angle is close to 1. It appears because the cart's acceleration affects the pole through the horizontal part of its direction.

In [ ]:
# Step 9.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.6: type this

```python
print("force", force)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `force` → *float (decimal number)*: `10.0`

**Prints:**
```text
force 10.0
```

In [ ]:
# Step 9.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.7: type this

```python
print("sin, cos", sin_t, cos_t)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `sin_t` → *float (decimal number)*: `0.04997916927067833`
2. `cos_t` → *float (decimal number)*: `0.9987502603949663`

**Prints:**
```text
sin, cos 0.04997916927067833 0.9987502603949663
```

In [ ]:
# Step 9.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.8: type this

```python
# cart acceleration if the pole were not there (Newton: a = F / m), plus a spin term
temp = (force + POLE_MASS_LEN * theta_dot ** 2 * sin_t) / TOTAL_MASS
```

- **Creates `temp`** — type: **float (decimal number)**; value: `9.09090909090909`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `force` → *float (decimal number)*: `10.0`
2. `POLE_MASS_LEN` → *float (decimal number)*: `0.05`
3. `theta_dot` → *float (decimal number)*: `0.0`
4. `theta_dot ** 2` → *float (decimal number)*: `0.0`
5. `POLE_MASS_LEN * theta_dot ** 2` → *float (decimal number)*: `0.0`
6. `sin_t` → *float (decimal number)*: `0.04997916927067833`
7. `POLE_MASS_LEN * theta_dot ** 2 * sin_t` → *float (decimal number)*: `0.0`
8. `force + POLE_MASS_LEN * theta_dot ** 2 * sin_t` → *float (decimal number)*: `10.0`
9. `TOTAL_MASS` → *float (decimal number)*: `1.1`

- **Why / what it means:** Python works out brackets first, then `**` (power), then `*` and `/`, then `+`: just like arithmetic at school. The evaluation list above shows every intermediate result in that order. `theta_dot ** 2` is the spin squared (0 here, since the pole is not spinning). The whole expression is **force divided by mass** (Newton's second law, `a = F / m`), plus a correction term for the pole's spin that happens to be zero right now. It is the cart's acceleration if the pole were not pushing back.

In [ ]:
# Step 9.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.9: type this

```python
print("temp", temp)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `temp` → *float (decimal number)*: `9.09090909090909`

**Prints:**
```text
temp 9.09090909090909
```

In [ ]:
# Step 9.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.10: type this

```python
# how quickly the pole's spin changes (gravity tips it, the cart's motion fights it)
theta_acc = (GRAVITY * sin_t - cos_t * temp) / (
    HALF_POLE * (4.0 / 3.0 - MASS_POLE * cos_t ** 2 / TOTAL_MASS))
```

- **Creates `theta_acc`** — type: **float (decimal number)**; value: `-13.824878764357754`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `GRAVITY` → *float (decimal number)*: `9.8`
2. `sin_t` → *float (decimal number)*: `0.04997916927067833`
3. `GRAVITY * sin_t` → *float (decimal number)*: `0.4897958588526477`
4. `cos_t` → *float (decimal number)*: `0.9987502603949663`
5. `temp` → *float (decimal number)*: `9.09090909090909`
6. `cos_t * temp` → *float (decimal number)*: `9.07954782177242`
7. `GRAVITY * sin_t - cos_t * temp` → *float (decimal number)*: `-8.589751962919772`
8. `HALF_POLE` → *float (decimal number)*: `0.5`
9. `4.0 / 3.0` → *float (decimal number)*: `1.3333333333333333`
10. `MASS_POLE` → *float (decimal number)*: `0.1`
11. `cos_t ** 2` → *float (decimal number)*: `0.997502082639013`
12. `MASS_POLE * cos_t ** 2` → *float (decimal number)*: `0.0997502082639013`
13. `TOTAL_MASS` → *float (decimal number)*: `1.1`
14. `MASS_POLE * cos_t ** 2 / TOTAL_MASS` → *float (decimal number)*: `0.09068200751263754`

- **Why / what it means:** The pole's **angular acceleration**: how fast its spin changes. The top part is a contest between two effects: `GRAVITY * sin_t` (gravity tipping the pole over, positive means rightward) and `cos_t * temp` (the cart accelerating out from under the pole). The first is 0.49, the second 9.08, so the second wins easily and the result is **negative**: pushing right makes the pole start tipping *left*. The bottom part is the pole's resistance to rotating.

In [ ]:
# Step 9.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.11: type this

```python
print("theta_acc", theta_acc)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `theta_acc` → *float (decimal number)*: `-13.824878764357754`

**Prints:**
```text
theta_acc -13.824878764357754
```

In [ ]:
# Step 9.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.12: type this

```python
# the cart's real acceleration, corrected for the pole pushing back
x_acc = temp - POLE_MASS_LEN * theta_acc * cos_t / TOTAL_MASS
```

- **Creates `x_acc`** — type: **float (decimal number)**; value: `9.71852733026505`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `temp` → *float (decimal number)*: `9.09090909090909`
2. `POLE_MASS_LEN` → *float (decimal number)*: `0.05`
3. `theta_acc` → *float (decimal number)*: `-13.824878764357754`
4. `POLE_MASS_LEN * theta_acc` → *float (decimal number)*: `-0.6912439382178878`
5. `cos_t` → *float (decimal number)*: `0.9987502603949663`
6. `POLE_MASS_LEN * theta_acc * cos_t` → *float (decimal number)*: `-0.6903800632915574`
7. `TOTAL_MASS` → *float (decimal number)*: `1.1`
8. `POLE_MASS_LEN * theta_acc * cos_t / TOTAL_MASS` → *float (decimal number)*: `-0.6276182393559613`

- **Why / what it means:** The cart's true acceleration: `temp` corrected for the pole pulling back on it. Because `theta_acc` is negative, the subtracted term is negative, so the correction adds a little: 9.09 becomes 9.72.

In [ ]:
# Step 9.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.13: type this

```python
print("x_acc", x_acc)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `x_acc` → *float (decimal number)*: `9.71852733026505`

**Prints:**
```text
x_acc 9.71852733026505
```

In [ ]:
# Step 9.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.14: type this

```python
# Euler integration: new = old + (time step) * (rate of change)
new_state = [x + TAU * x_dot,
             x_dot + TAU * x_acc,
             theta + TAU * theta_dot,
             theta_dot + TAU * theta_acc]
```

- **Creates `new_state`** — type: **list of 4 items (each a float)**; value: `[0.0, 0.194370546605301, 0.05, -0.2764975752871551]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `x` → *float (decimal number)*: `0.0`
2. `TAU` → *float (decimal number)*: `0.02`
3. `x_dot` → *float (decimal number)*: `0.0`
4. `TAU * x_dot` → *float (decimal number)*: `0.0`
5. `x + TAU * x_dot` → *float (decimal number)*: `0.0`
6. `x_acc` → *float (decimal number)*: `9.71852733026505`
7. `TAU * x_acc` → *float (decimal number)*: `0.194370546605301`
8. `x_dot + TAU * x_acc` → *float (decimal number)*: `0.194370546605301`
9. `theta` → *float (decimal number)*: `0.05`
10. `theta_dot` → *float (decimal number)*: `0.0`
11. `TAU * theta_dot` → *float (decimal number)*: `0.0`
12. `theta + TAU * theta_dot` → *float (decimal number)*: `0.05`
13. `theta_acc` → *float (decimal number)*: `-13.824878764357754`
14. `TAU * theta_acc` → *float (decimal number)*: `-0.2764975752871551`

- **Why / what it means:** **Euler integration**, the simplest way to move a simulation forward in time: *new value = old value + time step x rate of change*. A car at 100 m moving at 20 m/s is at 100 + 0.02 x 20 = 100.4 m after 0.02 s. We do this for each of the four state numbers, using that number's rate of change (position changes at the velocity, velocity at the acceleration, angle at the spin, spin at the angular acceleration). The result is a **new list of four floats**: the state one tick later.

In [ ]:
# Step 9.14: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.15: type this

```python
print(new_state)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `new_state` → *list of 4 items (each a float)*: `[0.0, 0.194370546605301, 0.05, -0.2764975752871551]`

**Prints:**
```text
[0.0, 0.194370546605301, 0.05, -0.2764975752871551]
```

In [ ]:
# Step 9.15: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9 complete: everything it printed, and what it means

```text
force 10.0
sin, cos 0.04997916927067833 0.9987502603949663
temp 9.09090909090909
theta_acc -13.824878764357754
x_acc 9.71852733026505
[0.0, 0.194370546605301, 0.05, -0.2764975752871551]
```

The push is `10.0`. For a 0.05 rad pole, `sin` is `0.04998` and `cos` is `0.99875`, nearly 1 (for small angles `sin(theta)` is almost `theta`). `temp` is `9.09`. The key number is `theta_acc = -13.82`: it is **negative**, so pushing right makes the pole start tipping *left*, even though gravity was tipping it right. Pushing the cart out from under a pole makes it lag behind, which is the opposite of what intuition about "pushing" suggests, and is precisely why balancing is hard. `x_acc` is `9.72`. In the new state, position is unchanged (velocity was 0), velocity is `0.02 x 9.72 = 0.194`, angle is unchanged (spin was 0), and spin is `0.02 x -13.82 = -0.276`.

**Try this yourself.** Change `action = 1` to `action = 0` and predict the signs of `temp`, `theta_acc` and `x_acc` before running.

## Step 10: The step function, and watching the pole fall

**The idea first.** We now wrap Step 9's lines in a function, `step(state, action)`, which is the environment's transition function: the rule for "what happens next". The agent will call it thousands of times. We test it, then watch what happens if we do nothing useful.

This step has **6 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 10.1: type this

```python
def step(state, action):
    x, x_dot, theta, theta_dot = state
    force = FORCE if action == 1 else -FORCE
    sin_t = math.sin(theta)
    cos_t = math.cos(theta)
    temp = (force + POLE_MASS_LEN * theta_dot ** 2 * sin_t) / TOTAL_MASS
    theta_acc = (GRAVITY * sin_t - cos_t * temp) / (
        HALF_POLE * (4.0 / 3.0 - MASS_POLE * cos_t ** 2 / TOTAL_MASS))
    x_acc = temp - POLE_MASS_LEN * theta_acc * cos_t / TOTAL_MASS
    return [x + TAU * x_dot,
            x_dot + TAU * x_acc,
            theta + TAU * theta_dot,
            theta_dot + TAU * theta_acc]
```

- **Creates** `step` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** This is the environment's **transition function**: it takes the current situation and a choice and returns the next situation. It is Step 9's code wrapped up so we can call it thousands of times. It has two parameters, `state` (a list of four floats) and `action` (0 or 1), and returns a new list of four floats.

In [ ]:
# Step 10.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `step([0.0, 0.0, 0.05, 0.0], 1)`

**What the function receives** (its inputs, called *parameters*):
- `state` — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.05, 0.0]`
- `action` — type: **int (whole number)**; value: `1`

**Now each line runs, top to bottom:**

```python
x, x_dot, theta, theta_dot = state
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.05, 0.0]`
- **Now `x` holds** — type: **float (decimal number)**; value: `0.0`
- **Now `x_dot` holds** — type: **float (decimal number)**; value: `0.0`
- **Now `theta` holds** — type: **float (decimal number)**; value: `0.05`
- **Now `theta_dot` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** The function's first act is to unpack its input list into four named floats, as in Step 9.

```python
force = FORCE if action == 1 else -FORCE
```
Evaluated inside-out (the main pieces):
1. `action` → *int (whole number)*: `1`
2. `action == 1` → *bool (True/False)*: `True`
3. `FORCE` → *float (decimal number)*: `10.0`
- **Now `force` holds** — type: **float (decimal number)**; value: `10.0`
- **Why:** Chooses the push direction. Here `action` is 1, so the force is +10.

```python
sin_t = math.sin(theta)
```
Evaluated inside-out (the main pieces):
1. `theta` → *float (decimal number)*: `0.05`
- **Now `sin_t` holds** — type: **float (decimal number)**; value: `0.04997916927067833`
- **Why:** Sine of the angle in radians.

```python
cos_t = math.cos(theta)
```
Evaluated inside-out (the main pieces):
1. `theta` → *float (decimal number)*: `0.05`
- **Now `cos_t` holds** — type: **float (decimal number)**; value: `0.9987502603949663`
- **Why:** Cosine of the angle.


#### …the same call, continued: `step([0.0, 0.0, 0.05, 0.0], 1)`

```python
temp = (force + POLE_MASS_LEN * theta_dot ** 2 * sin_t) / TOTAL_MASS
```
Evaluated inside-out (the main pieces):
1. `force` → *float (decimal number)*: `10.0`
2. `POLE_MASS_LEN` → *float (decimal number)*: `0.05`
3. `theta_dot` → *float (decimal number)*: `0.0`
4. `theta_dot ** 2` → *float (decimal number)*: `0.0`
5. `POLE_MASS_LEN * theta_dot ** 2` → *float (decimal number)*: `0.0`
6. `sin_t` → *float (decimal number)*: `0.04997916927067833`
7. `POLE_MASS_LEN * theta_dot ** 2 * sin_t` → *float (decimal number)*: `0.0`
8. `force + POLE_MASS_LEN * theta_dot ** 2 * sin_t` → *float (decimal number)*: `10.0`
- **Now `temp` holds** — type: **float (decimal number)**; value: `9.09090909090909`
- **Why:** Cart acceleration ignoring the pole (force over mass). Each intermediate result is shown above in the order Python computes it.

```python
theta_acc = (GRAVITY * sin_t - cos_t * temp) / (
    HALF_POLE * (4.0 / 3.0 - MASS_POLE * cos_t ** 2 / TOTAL_MASS))
```
Evaluated inside-out (the main pieces):
1. `GRAVITY` → *float (decimal number)*: `9.8`
2. `sin_t` → *float (decimal number)*: `0.04997916927067833`
3. `GRAVITY * sin_t` → *float (decimal number)*: `0.4897958588526477`
4. `cos_t` → *float (decimal number)*: `0.9987502603949663`
5. `temp` → *float (decimal number)*: `9.09090909090909`
6. `cos_t * temp` → *float (decimal number)*: `9.07954782177242`
7. `GRAVITY * sin_t - cos_t * temp` → *float (decimal number)*: `-8.589751962919772`
8. `HALF_POLE` → *float (decimal number)*: `0.5`
- **Now `theta_acc` holds** — type: **float (decimal number)**; value: `-13.824878764357754`
- **Why:** The pole's angular acceleration. Note the statement spans two source lines, but Python treats it as one statement because of the open bracket.

```python
x_acc = temp - POLE_MASS_LEN * theta_acc * cos_t / TOTAL_MASS
```
Evaluated inside-out (the main pieces):
1. `temp` → *float (decimal number)*: `9.09090909090909`
2. `POLE_MASS_LEN` → *float (decimal number)*: `0.05`
3. `theta_acc` → *float (decimal number)*: `-13.824878764357754`
4. `POLE_MASS_LEN * theta_acc` → *float (decimal number)*: `-0.6912439382178878`
5. `cos_t` → *float (decimal number)*: `0.9987502603949663`
6. `POLE_MASS_LEN * theta_acc * cos_t` → *float (decimal number)*: `-0.6903800632915574`
7. `TOTAL_MASS` → *float (decimal number)*: `1.1`
8. `POLE_MASS_LEN * theta_acc * cos_t / TOTAL_MASS` → *float (decimal number)*: `-0.6276182393559613`
- **Now `x_acc` holds** — type: **float (decimal number)**; value: `9.71852733026505`
- **Why:** The cart's corrected acceleration.

```python
return [x + TAU * x_dot,
        x_dot + TAU * x_acc,
        theta + TAU * theta_dot,
        theta_dot + TAU * theta_acc]
```
Evaluated inside-out (the main pieces):
1. `x` → *float (decimal number)*: `0.0`
2. `TAU` → *float (decimal number)*: `0.02`
3. `x_dot` → *float (decimal number)*: `0.0`
4. `TAU * x_dot` → *float (decimal number)*: `0.0`
5. `x + TAU * x_dot` → *float (decimal number)*: `0.0`
6. `x_acc` → *float (decimal number)*: `9.71852733026505`
7. `TAU * x_acc` → *float (decimal number)*: `0.194370546605301`
8. `x_dot + TAU * x_acc` → *float (decimal number)*: `0.194370546605301`
- **Why:** The function ends by building a **new list of four floats** and handing it back with `return`. This applies Euler integration to each state number: *new = old + TAU x rate of change*. The evaluation list shows each of the eight small calculations; the four results become the next state. The original `state` list that was passed in is **not modified**: we return a brand-new list.

**The function hands back** (its *return value*) — type: **list of 4 items (each a float)**; value: `[0.0, 0.194370546605301, 0.05, -0.2764975752871551]`


### Step 10.2: type this

```python
start = [0.0, 0.0, 0.0, 0.0]
```

- **Creates `start`** — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.0, 0.0]`

- **Why / what it means:** A perfectly upright, motionless pole. It makes a clean test, because any change in the next state must be caused by the push alone.

In [ ]:
# Step 10.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.3: type this

```python
print("push left :", step(start, 0))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `start` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.0, 0.0]`
2. `step(start, 0)` → *list of 4 items (each a float)*: `[0.0, -0.1951219512195122, 0.0, 0.2926829268292683]`

**Prints:**
```text
push left : [0.0, -0.1951219512195122, 0.0, 0.2926829268292683]
```

In [ ]:
# Step 10.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.4: type this

```python
print("push right:", step(start, 1))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `start` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.0, 0.0]`
2. `step(start, 1)` → *list of 4 items (each a float)*: `[0.0, 0.1951219512195122, 0.0, -0.2926829268292683]`

**Prints:**
```text
push right: [0.0, 0.1951219512195122, 0.0, -0.2926829268292683]
```

In [ ]:
# Step 10.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.5: type this

```python
state = [0.0, 0.0, 0.0, 0.0]
```

- **Creates `state`** — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.0, 0.0]`

- **Why / what it means:** Restarting from the upright state for the 'always push left' experiment.

In [ ]:
# Step 10.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.6: type this

```python
for tick in range(1, 11):
    state = step(state, 0)                # always push left
    print(tick, "x =", round(state[0], 4), " theta =", round(state[2], 4))
```

- **Creates `tick`** — type: **int (whole number)**; value: `10`

- **Creates `state`** — type: **list of 4 items (each a float)**; value: `[-0.17585859454842806, -1.9562338250740894, 0.2707437797652659, 3.127667606081298]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `range(1, 11)` → *range*: `range(1, 11)`
2. `state` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.0, 0.0]` *(evaluated 10 times; the first is shown)*
3. `step(state, 0)` → *list of 4 items (each a float)*: `[0.0, -0.1951219512195122, 0.0, 0.2926829268292683]` *(evaluated 10 times; the first is shown)*
4. `tick` → *int (whole number)*: `1` *(evaluated 10 times; the first is shown)*
5. `state[0]` → *float (decimal number)*: `0.0` *(evaluated 10 times; the first is shown)*
6. `round(state[0], 4)` → *float (decimal number)*: `0.0` *(evaluated 10 times; the first is shown)*
7. `state[2]` → *float (decimal number)*: `0.0` *(evaluated 10 times; the first is shown)*
8. `round(state[2], 4)` → *float (decimal number)*: `0.0` *(evaluated 10 times; the first is shown)*

**Prints:**
```text
1 x = 0.0  theta = 0.0
2 x = -0.0039  theta = 0.0059
3 x = -0.0117  theta = 0.0176
4 x = -0.0234  theta = 0.0352
5 x = -0.039  theta = 0.0587
6 x = -0.0586  theta = 0.0884
7 x = -0.082  theta = 0.1242
8 x = -0.1094  theta = 0.1664
9 x = -0.1407  theta = 0.2152
10 x = -0.1759  theta = 0.2707
```

- **Why / what it means:** A loop that runs ten times with `tick` taking the values 1 to 10 (`range(1, 11)` stops before 11). Each pass feeds the *previous result back in*: `state = step(state, 0)` reads the old `state`, computes the next one, and stores it under the same name. So `state` is a different list on every pass. `state[0]` and `state[2]` are position and angle; `round(value, 4)` trims the printout to four decimals.

In [ ]:
# Step 10.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10 complete: everything it printed, and what it means

```text
push left : [0.0, -0.1951219512195122, 0.0, 0.2926829268292683]
push right: [0.0, 0.1951219512195122, 0.0, -0.2926829268292683]
1 x = 0.0  theta = 0.0
2 x = -0.0039  theta = 0.0059
3 x = -0.0117  theta = 0.0176
4 x = -0.0234  theta = 0.0352
5 x = -0.039  theta = 0.0587
6 x = -0.0586  theta = 0.0884
7 x = -0.082  theta = 0.1242
8 x = -0.1094  theta = 0.1664
9 x = -0.1407  theta = 0.2152
10 x = -0.1759  theta = 0.2707
```

Pushing left from rest gives the cart velocity `-0.195` (moving left) and the pole spin `+0.293` (starting to lean right). Pushing right gives exactly the mirror image: the physics is symmetric, which is a useful sanity check. And once again the pole tips *opposite* to the push. In the loop, the angle grows `0.0059, 0.0176, 0.0352, 0.0587, ...`: the increases get bigger each tick, because a more tilted pole falls faster. At tick 9 the angle is `0.2152`, past the limit of `0.2094`, so a real episode would already be over. A pole left alone falls in under a fifth of a simulated second, so the controller must react constantly.

## Step 11: When does an episode end?

**The idea first.** The agent needs to know when it has failed. Failing is the only way an episode ends early.

This step has **4 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 11.1: type this

```python
def is_done(state):
    return abs(state[0]) > X_LIMIT or abs(state[2]) > THETA_LIMIT
```

- **Creates** `is_done` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A function that answers a yes/no question. It returns a `bool` (`True` or `False`).

In [ ]:
# Step 11.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `is_done([0.0, 0.0, 0.25, 0.0])`

**What the function receives** (its inputs, called *parameters*):
- `state` — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.25, 0.0]`

**Now each line runs, top to bottom:**

```python
return abs(state[0]) > X_LIMIT or abs(state[2]) > THETA_LIMIT
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.25, 0.0]`
2. `state[0]` → *float (decimal number)*: `0.0`
3. `abs(state[0])` → *float (decimal number)*: `0.0`
4. `X_LIMIT` → *float (decimal number)*: `2.4`
5. `abs(state[0]) > X_LIMIT` → *bool (True/False)*: `False`
6. `state[2]` → *float (decimal number)*: `0.25`
7. `abs(state[2])` → *float (decimal number)*: `0.25`
8. `THETA_LIMIT` → *float (decimal number)*: `0.20943951023931953`
- **Why:** `state[0]` is the first list item, `x` (lists count from 0), and `state[2]` is `theta`. `abs` strips the sign, so the limit applies equally on both sides. `or` produces `True` if *either* side is `True`; Python evaluates the left side first, and if it is already `True` it does not even look at the right (this is called short-circuiting). For the test state the cart is fine but the pole is past 12 degrees, so the answer is `True`.

**The function hands back** (its *return value*) — type: **bool (True/False)**; value: `True`


### Step 11.2: type this

```python
print(is_done([0.0, 0.0, 0.0, 0.0]))      # upright, centred
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `is_done([0.0, 0.0, 0.0, 0.0])` → *bool (True/False)*: `False`

**Prints:**
```text
False
```

In [ ]:
# Step 11.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.3: type this

```python
print(is_done([0.0, 0.0, 0.25, 0.0]))     # pole leaning 14 degrees
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `is_done([0.0, 0.0, 0.25, 0.0])` → *bool (True/False)*: `True`

**Prints:**
```text
True
```

In [ ]:
# Step 11.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.4: type this

```python
print(is_done([2.5, 0.0, 0.0, 0.0]))      # cart past the edge
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `is_done([2.5, 0.0, 0.0, 0.0])` → *bool (True/False)*: `True`

**Prints:**
```text
True
```

In [ ]:
# Step 11.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11 complete: everything it printed, and what it means

```text
False
True
True
```

`False` for an upright, centred pole; `True` for a pole at 0.25 rad (about 14 degrees); `True` for a cart at 2.5 m, past the 2.4 m edge.

## Step 12: Playing whole episodes with a policy

**The idea first.** A **policy** is any rule that takes a state and returns an action. This is the central object of reinforcement learning: learning *is* the search for a good policy. We write two simple ones to set a baseline that the learned agent must beat.

This step has **9 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 12.1: type this

```python
def run_episode(policy):
    state = reset()
    steps = 0
    while not is_done(state) and steps < MAX_STEPS:
        action = policy(state)            # ask the policy what to do
        state = step(state, action)       # the environment responds
        steps += 1                        # +1 reward for surviving this tick
    return steps
```

- **Creates** `run_episode` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Its parameter `policy` is itself a **function**: in Python you can pass a function to another function, like any other value. This lets the same code play episodes with any rule we like.

In [ ]:
# Step 12.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `run_episode(rule_policy)`

**What the function receives** (its inputs, called *parameters*):
- `policy` — type: **function (a reusable recipe)**; value: `<function rule_policy>`

**Now each line runs, top to bottom:**

```python
state = reset()
```
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[-0.049885718068557176, -0.0006422133534675356, 0.03676027754927809, -0.025608912311286803]`
- **Why:** Start a new episode: `reset()` returns a fresh list of four small random floats.

```python
steps = 0
```
- **Now `steps` holds** — type: **int (whole number)**; value: `0`
- **Why:** A counter, an `int`. The number of ticks survived is our score, since the reward is +1 per tick.

```python
while not is_done(state) and steps < MAX_STEPS:
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.049885718068557176, -0.0006422133534675356, 0.03676027754927809, -0.025608912311286803]`
2. `is_done(state)` → *bool (True/False)*: `False`
3. `not is_done(state)` → *bool (True/False)*: `True`
4. `steps` → *int (whole number)*: `0`
5. `MAX_STEPS` → *int (whole number)*: `500`
6. `steps < MAX_STEPS` → *bool (True/False)*: `True`
7. `not is_done(state) and steps < MAX_STEPS` → *bool (True/False)*: `True`
- **Why:** The loop keeps running **while** the condition is true: `not is_done(state)` is `True` as long as the episode has not failed, and `and steps < MAX_STEPS` also requires fewer than 500 ticks. Each pass the condition is re-checked with the *current* state.


#### …the same call, continued: `run_episode(rule_policy)`

```python
action = policy(state)            # ask the policy what to do
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.049885718068557176, -0.0006422133534675356, 0.03676027754927809, -0.025608912311286803]`
- **Now `action` holds** — type: **int (whole number)**; value: `0`
- **Why:** Calls whichever function was passed in. For `rule_policy` the answer depends on the state; for `random_policy` it does not.

```python
state = step(state, action)       # the environment responds
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.049885718068557176, -0.0006422133534675356, 0.03676027754927809, -0.025608912311286803]`
2. `action` → *int (whole number)*: `0`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[-0.04989856233562653, -0.19627152790830166, 0.03624809930305235, 0.2784419014830413]`
- **Why:** Calls the physics. The right side uses the *old* `state`; the result is stored back under the same name, replacing it. This is how the simulation moves forward one tick.

```python
steps += 1                        # +1 reward for surviving this tick
```
- **Now `steps` holds** — type: **int (whole number)**; value: `1`
- **Why:** `+=` adds to the existing value (`steps = steps + 1`). One more tick survived, so one more point of reward.

```python
while not is_done(state) and steps < MAX_STEPS:
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.04989856233562653, -0.19627152790830166, 0.03624809930305235, 0.2784419014830413]`
2. `is_done(state)` → *bool (True/False)*: `False`
3. `not is_done(state)` → *bool (True/False)*: `True`
4. `steps` → *int (whole number)*: `1`
5. `MAX_STEPS` → *int (whole number)*: `500`
6. `steps < MAX_STEPS` → *bool (True/False)*: `True`
7. `not is_done(state) and steps < MAX_STEPS` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
action = policy(state)            # ask the policy what to do
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.04989856233562653, -0.19627152790830166, 0.03624809930305235, 0.2784419014830413]`
- **Now `action` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `run_episode(rule_policy)`

```python
state = step(state, action)       # the environment responds
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.04989856233562653, -0.19627152790830166, 0.03624809930305235, 0.2784419014830413]`
2. `action` → *int (whole number)*: `1`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[-0.05382399289379256, -0.0016849305938261006, 0.04181693733271318, -0.0025916537687704633]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
steps += 1                        # +1 reward for surviving this tick
```
- **Now `steps` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
while not is_done(state) and steps < MAX_STEPS:
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05382399289379256, -0.0016849305938261006, 0.04181693733271318, -0.0025916537687704633]`
2. `is_done(state)` → *bool (True/False)*: `False`
3. `not is_done(state)` → *bool (True/False)*: `True`
4. `steps` → *int (whole number)*: `2`
5. `MAX_STEPS` → *int (whole number)*: `500`
6. `steps < MAX_STEPS` → *bool (True/False)*: `True`
7. `not is_done(state) and steps < MAX_STEPS` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
action = policy(state)            # ask the policy what to do
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05382399289379256, -0.0016849305938261006, 0.04181693733271318, -0.0025916537687704633]`
- **Now `action` holds** — type: **int (whole number)**; value: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
state = step(state, action)       # the environment responds
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05382399289379256, -0.0016849305938261006, 0.04181693733271318, -0.0025916537687704633]`
2. `action` → *int (whole number)*: `0`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[-0.05385769150566908, -0.19738087393549952, 0.04176510425733777, 0.3029862417810691]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `run_episode(rule_policy)`

```python
steps += 1                        # +1 reward for surviving this tick
```
- **Now `steps` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
while not is_done(state) and steps < MAX_STEPS:
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05385769150566908, -0.19738087393549952, 0.04176510425733777, 0.3029862417810691]`
2. `is_done(state)` → *bool (True/False)*: `False`
3. `not is_done(state)` → *bool (True/False)*: `True`
4. `steps` → *int (whole number)*: `3`
5. `MAX_STEPS` → *int (whole number)*: `500`
6. `steps < MAX_STEPS` → *bool (True/False)*: `True`
7. `not is_done(state) and steps < MAX_STEPS` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
action = policy(state)            # ask the policy what to do
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05385769150566908, -0.19738087393549952, 0.04176510425733777, 0.3029862417810691]`
- **Now `action` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
state = step(state, action)       # the environment responds
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[-0.05385769150566908, -0.19738087393549952, 0.04176510425733777, 0.3029862417810691]`
2. `action` → *int (whole number)*: `1`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[-0.05780530898437907, -0.002878270284626283, 0.047824829092959154, 0.02376212720033921]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 12.2: type this

```python
def random_policy(state):
    return random.randrange(2)            # ignores the state entirely
```

- **Creates** `random_policy` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A policy that ignores its input and returns 0 or 1 at random. It is our baseline: what you get with no intelligence at all.

In [ ]:
# Step 12.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.3: type this

```python
def rule_policy(state):
    return 1 if state[3] > 0 else 0       # push the way the pole is spinning
```

- **Creates** `rule_policy` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The one-line rule. `state[3]` is `theta_dot`, the pole's spin. A spin greater than 0 means the pole is rotating rightward, so we push right (1); otherwise left (0). The cart moves under the falling pole. The conditional expression `1 if ... else 0` evaluates to a `bool` first (in the trace, `True` for spin 0.3), then picks the value.

In [ ]:
# Step 12.3: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `rule_policy([0.0, 0.0, 0.05, 0.3])`

**What the function receives** (its inputs, called *parameters*):
- `state` — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.05, 0.3]`

**Now each line runs, top to bottom:**

```python
return 1 if state[3] > 0 else 0       # push the way the pole is spinning
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.05, 0.3]`
2. `state[3]` → *float (decimal number)*: `0.3`
3. `state[3] > 0` → *bool (True/False)*: `True`
4. `1 if state[3] > 0 else 0` → *int (whole number)*: `1`
- **Why:** `state[3] > 0` is a `bool`. For the test state (spin 0.3) it is `True`, so the function returns 1: push right.

**The function hands back** (its *return value*) — type: **int (whole number)**; value: `1`


### Step 12.4: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Second seed for the rule's test: the same value as before, so the starting states are identical and the comparison is fair.

In [ ]:
# Step 12.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.5: type this

```python
scores = [run_episode(random_policy) for _ in range(10)]
```

- **Creates `scores`** — type: **list of 10 items (each an int)**; value: `[11, 62, 37, 55, 12, 22, 22, 11, 47, 23]`

- **Why / what it means:** A list comprehension that plays ten full episodes and collects each result (a whole number of ticks) into a list of ten `int`s. We hand `run_episode` the function `random_policy` itself, without brackets, so it is passed rather than called.

In [ ]:
# Step 12.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.6: type this

```python
print("random:", scores, sum(scores) / len(scores))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *list of 10 items (each an int)*: `[11, 62, 37, 55, 12, 22, 22, 11, 47, 23]`
2. `sum(scores)` → *int (whole number)*: `302`
3. `len(scores)` → *int (whole number)*: `10`
4. `sum(scores) / len(scores)` → *float (decimal number)*: `30.2`

**Prints:**
```text
random: [11, 62, 37, 55, 12, 22, 22, 11, 47, 23] 30.2
```

In [ ]:
# Step 12.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.7: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Resets the random sequence so the ten episodes are repeatable.

In [ ]:
# Step 12.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.8: type this

```python
scores = [run_episode(rule_policy) for _ in range(10)]
```

- **Creates `scores`** — type: **list of 10 items (each an int)**; value: `[265, 173, 178, 258, 170, 165, 288, 253, 264, 183]`

- **Why / what it means:** The same, with the rule. Same seeds means the same starting states, so this is a fair comparison.

In [ ]:
# Step 12.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.9: type this

```python
print("rule  :", scores, sum(scores) / len(scores))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *list of 10 items (each an int)*: `[265, 173, 178, 258, 170, 165, 288, 253, 264, 183]`
2. `sum(scores)` → *int (whole number)*: `2197`
3. `len(scores)` → *int (whole number)*: `10`
4. `sum(scores) / len(scores)` → *float (decimal number)*: `219.7`

**Prints:**
```text
rule  : [265, 173, 178, 258, 170, 165, 288, 253, 264, 183] 219.7
```

In [ ]:
# Step 12.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12 complete: everything it printed, and what it means

```text
random: [11, 62, 37, 55, 12, 22, 22, 11, 47, 23] 30.2
rule  : [265, 173, 178, 258, 170, 165, 288, 253, 264, 183] 219.7
```

Random play survives about **30 ticks** (0.6 seconds). The one-line rule survives about **220**, seven times better. A human wrote that rule using physical insight. The aim of reinforcement learning is to have the *agent* discover a rule like this on its own, from nothing but rewards.

**Try this yourself.** Change the rule to `1 if state[2] > 0 else 0` (use the angle instead of the spin). Predict whether it does better or worse, then run.

---
# Part C: Q-learning

Now the learning itself. First the three ideas it rests on (reward and return, continuous versus discrete states, and the table), then the update rule, then the loop that applies it thousands of times.

## Step 13: Reward, return and the discount factor

**The idea first.** The agent does not just want the reward from the next tick; it wants the *total reward from now until the end*. That total, with future rewards counted a little less than immediate ones, is the **return**. How much less is set by the **discount factor**, gamma (a number between 0 and 1). This step makes the idea concrete.

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 13.1: type this

```python
rewards = [1] * 100                       # a list of one hundred 1s: 100 ticks survived
```

- **Creates `rewards`** — type: **list of 100 items (each an int)**; value: `[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...`

- **Why / what it means:** A list repeated 100 times: `[1] * 100` is a **list of 100 `int`s**, all equal to 1. It stands for surviving 100 ticks, each paying +1. Multiplying a list by a whole number repeats it.

In [ ]:
# Step 13.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13.2: type this

```python
for gamma in (1.0, 0.99, 0.9, 0.5):
    total = 0.0
    for k, r in enumerate(rewards):       # enumerate gives (position, item) pairs
        total += (gamma ** k) * r         # the k-th reward is worth gamma**k of its face value
    print("gamma", gamma, "-> discounted total", round(total, 2))
```

- **Creates `gamma`** — type: **float (decimal number)**; value: `0.5`

- **Creates `total`** — type: **float (decimal number)**; value: `2.0`

- **Creates `k`** — type: **int (whole number)**; value: `99`

- **Creates `r`** — type: **int (whole number)**; value: `1`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `rewards` → *list of 100 items (each an int)*: `[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...` *(evaluated 4 times; the first is shown)*
2. `enumerate(rewards)` → *enumerate*: `<enumerate object at 0x7fa183458c70>` *(evaluated 4 times; the first is shown)*
3. `gamma` → *float (decimal number)*: `1.0` *(evaluated 400 times; the first is shown)*
4. `k` → *int (whole number)*: `0` *(evaluated 400 times; the first is shown)*
5. `gamma ** k` → *float (decimal number)*: `1.0` *(evaluated 400 times; the first is shown)*
6. `r` → *int (whole number)*: `1` *(evaluated 400 times; the first is shown)*
7. `gamma ** k * r` → *float (decimal number)*: `1.0` *(evaluated 400 times; the first is shown)*
8. `total` → *float (decimal number)*: `100.0` *(evaluated 4 times; the first is shown)*
9. `round(total, 2)` → *float (decimal number)*: `100.0` *(evaluated 4 times; the first is shown)*

**Prints:**
```text
gamma 1.0 -> discounted total 100.0
gamma 0.99 -> discounted total 63.4
gamma 0.9 -> discounted total 10.0
gamma 0.5 -> discounted total 2.0
```

- **Why / what it means:** A **nested loop**: the outer `for` visits four values of gamma, one after another (the tuple lists them). For each, the code resets `total` to 0.0, then the inner loop adds up the discounted rewards. `enumerate(rewards)` produces pairs `(k, r)`: the position `k` (0 to 99) and the item `r` (always 1). The expression `(gamma ** k) * r` is gamma raised to the power `k`, times the reward: the reward `k` ticks ahead is worth `gamma**k` of its face value. The evaluation list shows only the *first pass* of each loop; the loop body runs many times.

In [ ]:
# Step 13.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13 complete: everything it printed, and what it means

```text
gamma 1.0 -> discounted total 100.0
gamma 0.99 -> discounted total 63.4
gamma 0.9 -> discounted total 10.0
gamma 0.5 -> discounted total 2.0
```

With `gamma = 1.0` the discount does nothing and the total is 100. With `0.99` it is **63.4**, with `0.9` it is **10.0**, and with `0.5` only **2.0**. A useful fact: with +1 per tick forever, the total approaches `1 / (1 - gamma)`: 100 for 0.99, 10 for 0.9, 2 for 0.5. So gamma sets the agent's *horizon*, roughly how many ticks ahead it can see. A tiny gamma makes it short-sighted, which is why `gamma = 0.5` struggles later in Experiment 3: a pole that is already doomed 20 ticks before it falls looks fine to an agent that cannot see 20 ticks ahead.

## Step 14: Chopping continuous numbers into buckets

**The idea first.** The state is four *continuous* numbers: the angle could be 0.0523 or 0.0524 or anything between. A table needs a **finite** list of rows. The fix is **discretization**: divide each number's range into a few buckets, and treat every state in the same buckets as the same situation. Think of reading a thermometer to the nearest 10 degrees: 71 and 74 both become "70s". We choose 3 buckets for position, 3 for velocity, 6 for angle and 6 for spin (more for the quantities that matter most for balancing).

This step has **12 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 14.1: type this

```python
LOW  = np.array([-X_LIMIT, -3.0, -THETA_LIMIT, -3.5])
```

- **Creates `LOW`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[-2.4, -3., -0.2094, -3.5]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `X_LIMIT` → *float (decimal number)*: `2.4`
2. `-X_LIMIT` → *float (decimal number)*: `-2.4`
3. `THETA_LIMIT` → *float (decimal number)*: `0.20943951023931953`
4. `-THETA_LIMIT` → *float (decimal number)*: `-0.20943951023931953`
5. `[-X_LIMIT, -3.0, -THETA_LIMIT, -3.5]` → *list of 4 items (each a float)*: `[-2.4, -3.0, -0.20943951023931953, -3.5]`

- **Why / what it means:** An array of four floats: the smallest value we will distinguish for each state number, in the order `[x, x_dot, theta, theta_dot]`. Note `-X_LIMIT` negates a constant. Using an array lets us do arithmetic on all four numbers at once.

In [ ]:
# Step 14.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.2: type this

```python
HIGH = np.array([ X_LIMIT,  3.0,  THETA_LIMIT,  3.5])
```

- **Creates `HIGH`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[2.4, 3., 0.2094, 3.5]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `X_LIMIT` → *float (decimal number)*: `2.4`
2. `THETA_LIMIT` → *float (decimal number)*: `0.20943951023931953`
3. `[X_LIMIT, 3.0, THETA_LIMIT, 3.5]` → *list of 4 items (each a float)*: `[2.4, 3.0, 0.20943951023931953, 3.5]`

- **Why / what it means:** The largest values: the mirror image of `LOW`.

In [ ]:
# Step 14.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.3: type this

```python
BINS = np.array([3, 3, 6, 6])
```

- **Creates `BINS`** — type: **numpy array, shape (4,), holding whole numbers**; value: `[3, 3, 6, 6]`

- **Why / what it means:** How many buckets to use for each of the four numbers: 3 for position, 3 for velocity, 6 for angle, 6 for spin. Whole numbers, because we will count buckets.

In [ ]:
# Step 14.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.4: type this

```python
def discretize(state):
    ratio = (np.array(state) - LOW) / (HIGH - LOW)     # each number -> position in its range, 0..1
    index = (ratio * BINS).astype(int)                 # stretch to bin count, chop decimals
    index = np.clip(index, 0, BINS - 1)                # keep inside the valid bins
    return tuple(int(i) for i in index)                # plain ints in a tuple, usable as an address
```

- **Creates** `discretize` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Converts a *continuous* state (a list of four decimals) into a *discrete* one (a tuple of four whole numbers) that can address a table. Think of rounding a thermometer reading to the nearest ten degrees.

In [ ]:
# Step 14.4: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `discretize([0.0, 0.0, 0.1, -1.0])`

**What the function receives** (its inputs, called *parameters*):
- `state` — type: **list of 4 items (each a float)**; value: `[0.0, 0.0, 0.1, -1.0]`

**Now each line runs, top to bottom:**

```python
ratio = (np.array(state) - LOW) / (HIGH - LOW)     # each number -> position in its range, 0..1
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.1, -1.0]`
2. `np.array(state)` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0.1, -1.]`
3. `LOW` → *numpy array, shape (4,), holding decimal numbers*: `[-2.4, -3., -0.2094, -3.5]`
4. `np.array(state) - LOW` → *numpy array, shape (4,), holding decimal numbers*: `[2.4, 3., 0.3094, 2.5]`
5. `HIGH` → *numpy array, shape (4,), holding decimal numbers*: `[2.4, 3., 0.2094, 3.5]`
6. `HIGH - LOW` → *numpy array, shape (4,), holding decimal numbers*: `[4.8, 6., 0.4189, 7.]`
- **Now `ratio` holds** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.5, 0.5, 0.7387, 0.3571]`
- **Why:** Step 1: `np.array(state)` converts the list to an array so arithmetic works on all four numbers at once. Step 2: subtracting `LOW` measures each number *from the bottom of its own range*. Step 3: dividing by `HIGH - LOW` (the width of each range) rescales everything to 0 to 1. So `ratio` says *how far through its range* each number is: 0.5 means exactly halfway.


#### …the same call, continued: `discretize([0.0, 0.0, 0.1, -1.0])`

```python
index = (ratio * BINS).astype(int)                 # stretch to bin count, chop decimals
```
Evaluated inside-out (the main pieces):
1. `ratio` → *numpy array, shape (4,), holding decimal numbers*: `[0.5, 0.5, 0.7387, 0.3571]`
2. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
3. `ratio * BINS` → *numpy array, shape (4,), holding decimal numbers*: `[1.5, 1.5, 4.4324, 2.1429]`
- **Now `index` holds** — type: **numpy array, shape (4,), holding whole numbers**; value: `[1, 1, 4, 2]`
- **Why:** Multiplying by the bucket counts stretches 0-to-1 into 0-to-(number of buckets). `astype(int)` then **chops off the decimals**, which is the bucketing: 4.43 becomes bucket 4. Values in [4, 5) all land in bucket 4.

```python
index = np.clip(index, 0, BINS - 1)                # keep inside the valid bins
```
Evaluated inside-out (the main pieces):
1. `index` → *numpy array, shape (4,), holding whole numbers*: `[1, 1, 4, 2]`
2. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
3. `BINS - 1` → *numpy array, shape (4,), holding whole numbers*: `[2, 2, 5, 5]`
4. `np.clip(index, 0, BINS - 1)` → *numpy array, shape (4,), holding whole numbers*: `[1, 1, 4, 2]`
- **Why:** Guards against values outside the range. A ratio of exactly 1.0 (or anything beyond) would give a bucket number equal to the bucket count, which does not exist, so `clip` forces it into 0 to (buckets - 1). Here the numbers are already valid, so nothing changes.

```python
return tuple(int(i) for i in index)                # plain ints in a tuple, usable as an address
```
Evaluated inside-out (the main pieces):
1. `(int(i) for i in index)` → *generator*: `<generator object discretize.<locals>.<genexpr> at 0x7fa18343cd40>`
2. `tuple((int(i) for i in index))` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`
- **Why:** Two conversions: `int(i)` turns numpy's integer type into a plain Python `int`, and `tuple(...)` makes a **tuple**. A tuple of whole numbers is exactly what numpy accepts as a multi-dimensional index, so `Q[this_tuple]` will work.

**The function hands back** (its *return value*) — type: **tuple of 4 items (each an int)**; value: `(1, 1, 4, 2)`


### Step 14.5: type this

```python
print(discretize([0.0, 0.0, 0.0, 0.0]))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `discretize([0.0, 0.0, 0.0, 0.0])` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`

**Prints:**
```text
(1, 1, 3, 3)
```

In [ ]:
# Step 14.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.6: type this

```python
print(discretize([-2.0, 0.0, 0.0, 0.0]))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[-2.0, 0.0, 0.0, 0.0]` → *list of 4 items (each a float)*: `[-2.0, 0.0, 0.0, 0.0]`
2. `discretize([-2.0, 0.0, 0.0, 0.0])` → *tuple of 4 items (each an int)*: `(0, 1, 3, 3)`

**Prints:**
```text
(0, 1, 3, 3)
```

In [ ]:
# Step 14.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.7: type this

```python
print(discretize([0.0, 0.0, 0.1, -1.0]))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[0.0, 0.0, 0.1, -1.0]` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.1, -1.0]`
2. `discretize([0.0, 0.0, 0.1, -1.0])` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`

**Prints:**
```text
(1, 1, 4, 2)
```

In [ ]:
# Step 14.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.8: type this

```python
print(discretize([0.0, 0.0, 5.0, 0.0]))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `discretize([0.0, 0.0, 5.0, 0.0])` → *tuple of 4 items (each an int)*: `(1, 1, 5, 3)`

**Prints:**
```text
(1, 1, 5, 3)
```

In [ ]:
# Step 14.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.9: type this

```python
ratio = (np.array([0.0, 0.0, 0.1, -1.0]) - LOW) / (HIGH - LOW)
```

- **Creates `ratio`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.5, 0.5, 0.7387, 0.3571]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[0.0, 0.0, 0.1, -1.0]` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.1, -1.0]`
2. `np.array([0.0, 0.0, 0.1, -1.0])` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0.1, -1.]`
3. `LOW` → *numpy array, shape (4,), holding decimal numbers*: `[-2.4, -3., -0.2094, -3.5]`
4. `np.array([0.0, 0.0, 0.1, -1.0]) - LOW` → *numpy array, shape (4,), holding decimal numbers*: `[2.4, 3., 0.3094, 2.5]`
5. `HIGH` → *numpy array, shape (4,), holding decimal numbers*: `[2.4, 3., 0.2094, 3.5]`
6. `HIGH - LOW` → *numpy array, shape (4,), holding decimal numbers*: `[4.8, 6., 0.4189, 7.]`

- **Why / what it means:** The same arithmetic outside a function, for one example, so each intermediate array can be printed and checked by hand.

In [ ]:
# Step 14.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.10: type this

```python
print(ratio)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `ratio` → *numpy array, shape (4,), holding decimal numbers*: `[0.5, 0.5, 0.7387, 0.3571]`

**Prints:**
```text
[0.5        0.5        0.73873241 0.35714286]
```

In [ ]:
# Step 14.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.11: type this

```python
print(ratio * BINS)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `ratio` → *numpy array, shape (4,), holding decimal numbers*: `[0.5, 0.5, 0.7387, 0.3571]`
2. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
3. `ratio * BINS` → *numpy array, shape (4,), holding decimal numbers*: `[1.5, 1.5, 4.4324, 2.1429]`

**Prints:**
```text
[1.5        1.5        4.43239449 2.14285714]
```

In [ ]:
# Step 14.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.12: type this

```python
print((ratio * BINS).astype(int))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `ratio` → *numpy array, shape (4,), holding decimal numbers*: `[0.5, 0.5, 0.7387, 0.3571]`
2. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
3. `ratio * BINS` → *numpy array, shape (4,), holding decimal numbers*: `[1.5, 1.5, 4.4324, 2.1429]`
4. `(ratio * BINS).astype(int)` → *numpy array, shape (4,), holding whole numbers*: `[1, 1, 4, 2]`

**Prints:**
```text
[1 1 4 2]
```

In [ ]:
# Step 14.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14 complete: everything it printed, and what it means

```text
(1, 1, 3, 3)
(0, 1, 3, 3)
(1, 1, 4, 2)
(1, 1, 5, 3)
[0.5        0.5        0.73873241 0.35714286]
[1.5        1.5        4.43239449 2.14285714]
[1 1 4 2]
```

A perfectly upright, still, centred pole is bucket `(1, 1, 3, 3)`, the middle of everything. A cart at -2.0 m is in the left bucket 0. An angle of 5.0 radians is wildly out of range, but `clip` puts it in the last bucket, 5, instead of crashing. Check the arithmetic by hand for the angle 0.1: `(0.1 - (-0.2094)) / (0.2094 - (-0.2094)) = 0.3094 / 0.4189 = 0.7387`. Times 6 buckets is `4.43`, and `astype(int)` makes it bucket 4. Position 0.0 has ratio exactly 0.5, times 3 is 1.5, so bucket 1, the middle of buckets 0, 1, 2. In total there are 3 x 3 x 6 x 6 = **324** distinct situations.

**Try this yourself.** Compute by hand the bucket for spin = -1.0 (the fourth number): `(-1.0 + 3.5) / 7.0 = 0.357`, times 6 = 2.14, so bucket 2. The output agrees.

## Step 15: The Q-table: a five-dimensional block of numbers

**The idea first.** **Q(s, a)** (the "quality" of action `a` in situation `s`) is the agent's estimate of *the total future reward if it takes action `a` now and plays well afterwards*. We store these estimates in an array. With 324 situations and 2 actions that is 648 numbers. Because the situation is four bucket numbers, the array has five dimensions: four to select the situation and one to select the action.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 15.1: type this

```python
Q = np.zeros(tuple(BINS) + (2,))          # (3,3,6,6) + (2,) -> shape (3,3,6,6,2)
```

- **Creates `Q`** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
2. `tuple(BINS)` → *tuple of 4 items (each a numpy int)*: `(3, 3, 6, 6)`
3. `tuple(BINS) + (2,)` → *tuple of 5 items*: `(3, 3, 6, 6, 2)`

- **Why / what it means:** `tuple(BINS)` converts the array `[3 3 6 6]` into the tuple `(3, 3, 6, 6)`. Adding `(2,)` (a one-item tuple; the comma is required) **joins the tuples**, giving `(3, 3, 6, 6, 2)`. `np.zeros` builds an array of that shape filled with 0.0. Think of it as a block with four 'address' dimensions to pick a situation and a fifth, of size 2, to pick an action.

In [ ]:
# Step 15.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.2: type this

```python
print(Q.shape, Q.size)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
2. `Q.shape` → *tuple of 5 items (each an int)*: `(3, 3, 6, 6, 2)`
3. `Q.size` → *int (whole number)*: `648`

**Prints:**
```text
(3, 3, 6, 6, 2) 648
```

In [ ]:
# Step 15.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.3: type this

```python
s = discretize([0.0, 0.0, 0.1, -1.0])     # the "row address" of a state: (1, 1, 4, 2)
```

- **Creates `s`** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 4, 2)`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[0.0, 0.0, 0.1, -1.0]` → *list of 4 items (each a float)*: `[0.0, 0.0, 0.1, -1.0]`

- **Why / what it means:** `s` becomes the **address** of a situation: a tuple of four whole numbers. We reuse the example from Step 14.

In [ ]:
# Step 15.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.4: type this

```python
print(Q[s])                               # the two numbers: [push-left value, push-right value]
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`
3. `Q[s]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 0.]`

**Prints:**
```text
[0. 0.]
```

In [ ]:
# Step 15.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.5: type this

```python
Q[s + (1,)] = 5.0                         # s + (1,) extends the address with the action
```

- **Changes `Q` in place** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value now: `648 numbers; 1 are non-zero; smallest 0, largest 5`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`
2. `s + (1,)` → *tuple of 5 items (each an int)*: `(1, 1, 4, 2, 1)`

- **Why / what it means:** `s + (1,)` joins the situation's address with the action number, giving a 5-item tuple that selects **exactly one box**. We store 5.0 there. Only that box changes; its sibling for action 0 stays 0.0.

In [ ]:
# Step 15.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.6: type this

```python
print(Q[s])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 1 are non-zero; smallest 0, largest 5`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`
3. `Q[s]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 5.]`

**Prints:**
```text
[0. 5.]
```

In [ ]:
# Step 15.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.7: type this

```python
print(Q[s].max(), Q[s].argmax())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 1 are non-zero; smallest 0, largest 5`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 4, 2)`
3. `Q[s]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 5.]`
4. `Q[s].max()` → *float (decimal number)*: `5.0`
5. `Q[s].argmax()` → *numpy integer (a whole number)*: `1`

**Prints:**
```text
5.0 1
```

In [ ]:
# Step 15.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15 complete: everything it printed, and what it means

```text
(3, 3, 6, 6, 2) 648
[0. 0.]
[0. 5.]
5.0 1
```

The shape is `(3, 3, 6, 6, 2)` with `648` boxes. A fresh row is `[0. 0.]`: zero means "I know nothing yet", not "this is neutral". After storing 5.0 for action 1, the row is `[0. 5.]`, and `max` and `argmax` give `5.0 1` (push right is best).

## Step 16: Choosing actions: explore or exploit

**The idea first.** If the agent always picked the action that looks best now, it would never discover that something it has not tried is better. So with a small probability **epsilon** it picks a random action instead (**exploring**), otherwise it takes the best-known action (**exploiting**). This is called *epsilon-greedy*.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 16.1: type this

```python
def choose_action(Q, s, epsilon):
    if random.random() < epsilon:         # with probability epsilon: explore
        return random.randrange(2)
    row = Q[s]
    if row[0] == row[1]:                  # tie: do not always favour action 0
        return random.randrange(2)
    return int(np.argmax(row))            # otherwise exploit: the best-known action
```

- **Creates** `choose_action` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Its parameters: the table `Q`, the situation `s` (a tuple of four ints) and `epsilon` (a float between 0 and 1). It returns 0 or 1.

In [ ]:
# Step 16.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `choose_action(np.zeros((3, 3, 6, 6, 2)), (1, 1, 3, 3), 0.0)`

**What the function receives** (its inputs, called *parameters*):
- `Q` — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers, all zero`
- `s` — type: **tuple of 4 items (each an int)**; value: `(1, 1, 3, 3)`
- `epsilon` — type: **float (decimal number)**; value: `0.0`

**Now each line runs, top to bottom:**

```python
if random.random() < epsilon:         # with probability epsilon: explore
```
Evaluated inside-out (the main pieces):
1. `random.random()` → *float (decimal number)*: `0.23861592861522019`
2. `epsilon` → *float (decimal number)*: `0.0`
3. `random.random() < epsilon` → *bool (True/False)*: `False`
- **Why:** `random.random()` is a float in [0, 1). With epsilon 0.0 this is never true, so we skip the explore branch and go on to exploit.

```python
row = Q[s]
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`
- **Now `row` holds** — type: **numpy array, shape (2,), holding decimal numbers**; value: `[0., 0.]`
- **Why:** `Q[s]` with a 4-item address into a 5-dimensional array leaves one dimension open, so we get back a **1D array of two floats**: the value of each action in this situation. Careful: this is a *view* of the table, not a copy. Changing `row` would change `Q`.


#### …the same call, continued: `choose_action(np.zeros((3, 3, 6, 6, 2)), (1, 1, 3, 3), 0.0)`

```python
if row[0] == row[1]:                  # tie: do not always favour action 0
```
Evaluated inside-out (the main pieces):
1. `row` → *numpy array, shape (2,), holding decimal numbers*: `[0., 0.]`
2. `row[0]` → *float (decimal number)*: `0.0`
3. `row[1]` → *float (decimal number)*: `0.0`
4. `row[0] == row[1]` → *bool*: `np.True_`
- **Why:** Compares the two floats in the row. Both are 0.0 in an untrained table, so they are equal: this is a **tie**.

```python
return random.randrange(2)
```
Evaluated inside-out (the main pieces):
1. `random.randrange(2)` → *int (whole number)*: `0`
- **Why:** On a tie we return a random 0 or 1 instead of always favouring action 0.

**The function hands back** (its *return value*) — type: **int (whole number)**; value: `0`


### Step 16.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness for the tests.

In [ ]:
# Step 16.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.3: type this

```python
Q = np.zeros(tuple(BINS) + (2,))
```

- **Creates `Q`** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
2. `tuple(BINS)` → *tuple of 4 items (each a numpy int)*: `(3, 3, 6, 6)`
3. `tuple(BINS) + (2,)` → *tuple of 5 items*: `(3, 3, 6, 6, 2)`

- **Why / what it means:** A fresh, untrained table for the test.

In [ ]:
# Step 16.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.4: type this

```python
s = (1, 1, 3, 3)
```

- **Creates `s`** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 3, 3)`

- **Why / what it means:** A situation address written directly as a tuple of four ints: the middle of every range, i.e. a still, upright, centred pole.

In [ ]:
# Step 16.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.5: type this

```python
print([choose_action(Q, s, 0.0) for _ in range(10)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 0.0) for _ in range(10)]` → *list of 10 items (each an int)*: `[1, 1, 1, 1, 0, 1, 0, 1, 0, 0]`

**Prints:**
```text
[1, 1, 1, 1, 0, 1, 0, 1, 0, 0]
```

In [ ]:
# Step 16.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.6: type this

```python
Q[s + (1,)] = 5.0
```

- **Changes `Q` in place** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value now: `648 numbers; 1 are non-zero; smallest 0, largest 5`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`
2. `s + (1,)` → *tuple of 5 items (each an int)*: `(1, 1, 3, 3, 1)`

- **Why / what it means:** Now action 1 has a clearly higher value in this situation, so exploiting should choose it every time.

In [ ]:
# Step 16.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.7: type this

```python
print([choose_action(Q, s, 0.0) for _ in range(10)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 0.0) for _ in range(10)]` → *list of 10 items (each an int)*: `[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]`

**Prints:**
```text
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
```

In [ ]:
# Step 16.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.8: type this

```python
print([choose_action(Q, s, 0.5) for _ in range(10)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 0.5) for _ in range(10)]` → *list of 10 items (each an int)*: `[0, 1, 1, 1, 1, 1, 1, 1, 1, 0]`

**Prints:**
```text
[0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
```

In [ ]:
# Step 16.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16 complete: everything it printed, and what it means

```text
[1, 1, 1, 1, 0, 1, 0, 1, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
```

With an all-zero table every action ties, so the output is a random mix of 0s and 1s. After setting action 1 to 5.0, epsilon 0 gives ten 1s: the table decides every time. With epsilon 0.5, about half the choices are random, so a few 0s sneak in (two of ten here).

## Step 17: One learning update, by hand

**The idea first.** This is the whole of Q-learning, in five lines. We set up a situation where we know the numbers, so you can check the arithmetic yourself.

This step has **13 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 17.1: type this

```python
ALPHA = 0.2
GAMMA = 0.99
```

- **Creates `ALPHA`** — type: **float (decimal number)**; value: `0.2`

- **Why / what it means:** The **learning rate**, a float between 0 and 1: what fraction of the gap between the old belief and the new evidence we close each time.

- **Creates `GAMMA`** — type: **float (decimal number)**; value: `0.99`

- **Why / what it means:** The **discount factor**: how much a reward one tick later is worth.

In [ ]:
# Step 17.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.2: type this

```python
Q = np.zeros(tuple(BINS) + (2,))
```

- **Creates `Q`** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
2. `tuple(BINS)` → *tuple of 4 items (each a numpy int)*: `(3, 3, 6, 6)`
3. `tuple(BINS) + (2,)` → *tuple of 5 items*: `(3, 3, 6, 6, 2)`

- **Why / what it means:** A fresh table. Every box starts at 0.0.

In [ ]:
# Step 17.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.3: type this

```python
s_next = (1, 1, 3, 3)                     # pretend we landed here and it already has known values
```

- **Creates `s_next`** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 3, 3)`

- **Why / what it means:** A situation we pretend to land in. Its address is a tuple of four ints.

In [ ]:
# Step 17.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.4: type this

```python
Q[s_next + (0,)] = 10.0
```

- **Changes `Q` in place** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value now: `648 numbers; 1 are non-zero; smallest 0, largest 10`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`
2. `s_next + (0,)` → *tuple of 5 items (each an int)*: `(1, 1, 3, 3, 0)`

- **Why / what it means:** We set up a situation where we *know* the numbers, so the arithmetic is checkable: pushing left from there is believed to be worth 10.

In [ ]:
# Step 17.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.5: type this

```python
Q[s_next + (1,)] = 20.0
```

- **Changes `Q` in place** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value now: `648 numbers; 2 are non-zero; smallest 0, largest 20`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`
2. `s_next + (1,)` → *tuple of 5 items (each an int)*: `(1, 1, 3, 3, 1)`

- **Why / what it means:** ...and pushing right worth 20. The best option there is 20.

In [ ]:
# Step 17.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.6: type this

```python
s = (1, 1, 3, 2)                          # the state we were in
a = 1                                     # the action we took
reward = 1                                # survived one tick
```

- **Creates `s`** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 3, 2)`

- **Why / what it means:** The situation we were in *before* the move (its last number differs by one bucket).

- **Creates `a`** — type: **int (whole number)**; value: `1`

- **Why / what it means:** The action we took: push right.

- **Creates `reward`** — type: **int (whole number)**; value: `1`

- **Why / what it means:** The reward we received: one tick survived.

In [ ]:
# Step 17.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.7: type this

```python
old = Q[s + (a,)]                         # current belief
```

- **Creates `old`** — type: **float (decimal number)**; value: `0.0`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 2 are non-zero; smallest 0, largest 20`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 3, 2)`
3. `a` → *int (whole number)*: `1`
4. `(a,)` → *tuple of 1 item (each an int)*: `(1,)`
5. `s + (a,)` → *tuple of 5 items (each an int)*: `(1, 1, 3, 2, 1)`

- **Why / what it means:** Reads **one box**: the current belief about taking action `a` in situation `s`. It is a numpy float (0.0, since the table is untrained). This comes from the situation we *left*.

In [ ]:
# Step 17.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.8: type this

```python
best_next = Q[s_next].max()               # best belief about where we landed
```

- **Creates `best_next`** — type: **float (decimal number)**; value: `20.0`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 2 are non-zero; smallest 0, largest 20`
2. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 3, 3)`
3. `Q[s_next]` → *numpy array, shape (2,), holding decimal numbers*: `[10., 20.]`

- **Why / what it means:** `Q[s_next]` is the two-number row for the situation we *arrived at*; `.max()` picks the larger: 20.0. This is 'if I act well from where I landed, how much more will I get?'. It comes from the situation we *arrived at*, which is the key difference from `old`.

In [ ]:
# Step 17.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.9: type this

```python
target = reward + GAMMA * best_next       # what the belief should be, given what we saw
```

- **Creates `target`** — type: **float (decimal number)**; value: `20.8`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `reward` → *int (whole number)*: `1`
2. `GAMMA` → *float (decimal number)*: `0.99`
3. `best_next` → *float (decimal number)*: `20.0`
4. `GAMMA * best_next` → *float (decimal number)*: `19.8`

- **Why / what it means:** Our new opinion of what the box *should* hold: what we just got, plus the discounted best we expect afterwards. Order of operations: `GAMMA * best_next` first (0.99 x 20 = 19.8), then add the reward (1) to get 20.8.

In [ ]:
# Step 17.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.10: type this

```python
new = old + ALPHA * (target - old)        # move a fraction ALPHA of the way there
```

- **Creates `new`** — type: **float (decimal number)**; value: `4.16`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `old` → *float (decimal number)*: `0.0`
2. `ALPHA` → *float (decimal number)*: `0.2`
3. `target` → *float (decimal number)*: `20.8`
4. `target - old` → *float (decimal number)*: `20.8`
5. `ALPHA * (target - old)` → *float (decimal number)*: `4.16`

- **Why / what it means:** `target - old` is the **error**: how wrong the old belief was (20.8 here). We close a fraction ALPHA of it: 0.2 x 20.8 = 4.16, added to the old value 0.0. So the belief moves from 0 to 4.16, partway to 20.8, not all the way.

In [ ]:
# Step 17.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.11: type this

```python
print(old, best_next, target, new)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `old` → *float (decimal number)*: `0.0`
2. `best_next` → *float (decimal number)*: `20.0`
3. `target` → *float (decimal number)*: `20.8`
4. `new` → *float (decimal number)*: `4.16`

**Prints:**
```text
0.0 20.0 20.8 4.16
```

In [ ]:
# Step 17.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.12: type this

```python
Q[s + (a,)] = new
```

- **Changes `Q` in place** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value now: `648 numbers; 3 are non-zero; smallest 0, largest 20`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s` → *tuple of 4 items (each an int)*: `(1, 1, 3, 2)`
2. `a` → *int (whole number)*: `1`
3. `(a,)` → *tuple of 1 item (each an int)*: `(1,)`
4. `s + (a,)` → *tuple of 5 items (each an int)*: `(1, 1, 3, 2, 1)`

- **Why / what it means:** The only write to the table: one box. The row printed afterwards shows `[0. 4.16]`: the sibling box is untouched.

In [ ]:
# Step 17.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.13: type this

```python
print(Q[s])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 3 are non-zero; smallest 0, largest 20`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 3, 2)`
3. `Q[s]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 4.16]`

**Prints:**
```text
[0.   4.16]
```

In [ ]:
# Step 17.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17 complete: everything it printed, and what it means

```text
0.0 20.0 20.8 4.16
[0.   4.16]
```

`0.0 20.0 20.8 4.16`: old 0, best next 20, target `1 + 0.99 x 20 = 20.8`, new `0 + 0.2 x (20.8 - 0) = 4.16`. The row printed is `[0. 4.16]`: only the box for action 1 changed. The same number written another way: `new = (1 - ALPHA) x old + ALPHA x target`, a blend of old belief and new evidence, like a moving average.

**Try this yourself.** Run the update a second time (just repeat the lines from `old = ...` on) and predict the new value. (Target stays 20.8, old is now 4.16, so new = 4.16 + 0.2 x 16.64 = 7.488.)

## Step 18: The training loop

**The idea first.** Now we repeat the update thousands of times, from many different starts, while the agent explores and gradually exploits what it has learned.

This step has **5 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 18.1: type this

```python
def train(episodes, alpha=0.2, gamma=0.99, eps_start=1.0, eps_min=0.01, eps_decay=0.999):
    Q = np.zeros(tuple(BINS) + (2,))
    epsilon = eps_start
    history = []
    for episode in range(episodes):
        state = reset()
        s = discretize(state)
        steps = 0
        while True:
            a = choose_action(Q, s, epsilon)
            state = step(state, a)
            steps += 1
            failed = is_done(state)
            s_next = discretize(state)

            reward = 1
            best_next = 0.0 if failed else Q[s_next].max()   # no future after failing
            target = reward + gamma * best_next
            Q[s + (a,)] += alpha * (target - Q[s + (a,)])

            s = s_next
            if failed or steps >= MAX_STEPS:
                break
        epsilon = max(eps_min, epsilon * eps_decay)
        history.append(steps)
    return Q, history
```

- **Creates** `train` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The full algorithm in one function. It ends with `return Q, history`, which returns **two values** as a tuple; the caller unpacks them, as in `Q, history = train(300)`. The parameters after `episodes` have **default values** (like `alpha=0.2`), so `train(300)` works and you override only what you want. It returns two things: the learned table and the list of episode lengths.

In [ ]:
# Step 18.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `train(1)`

**What the function receives** (its inputs, called *parameters*):
- `episodes` — type: **int (whole number)**; value: `1`
- `alpha` — type: **float (decimal number)**; value: `0.2`
- `gamma` — type: **float (decimal number)**; value: `0.99`
- `eps_start` — type: **float (decimal number)**; value: `1.0`
- `eps_min` — type: **float (decimal number)**; value: `0.01`
- `eps_decay` — type: **float (decimal number)**; value: `0.999`

**Now each line runs, top to bottom:**

```python
Q = np.zeros(tuple(BINS) + (2,))
```
Evaluated inside-out (the main pieces):
1. `BINS` → *numpy array, shape (4,), holding whole numbers*: `[3, 3, 6, 6]`
2. `tuple(BINS)` → *tuple of 4 items (each a numpy int)*: `(3, 3, 6, 6)`
3. `tuple(BINS) + (2,)` → *tuple of 5 items*: `(3, 3, 6, 6, 2)`
- **Now `Q` holds** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers, all zero`
- **Why:** A fresh table of 648 zeros: the agent knows nothing.

```python
epsilon = eps_start
```
- **Now `epsilon` holds** — type: **float (decimal number)**; value: `1.0`
- **Why:** Epsilon starts at 1.0, meaning 'always explore': at first the table knows nothing, so acting randomly is as good as anything.

```python
history = []
```
- **Now `history` holds** — type: **list of 0 items**; value: `[]`
- **Why:** An empty **list**, which we will `append` to. It records how many ticks each episode lasted.

```python
for episode in range(episodes):
```
Evaluated inside-out (the main pieces):
1. `episodes` → *int (whole number)*: `1`
2. `range(episodes)` → *range*: `range(0, 1)`
- **Now `episode` holds** — type: **int (whole number)**; value: `0`
- **Why:** The outer loop: one pass per episode. We traced `train(1)`, so it runs once.


#### …the same call, continued: `train(1)`

```python
state = reset()
```
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[0.046035290461553954, 0.02189762200322956, -0.023633383322497127, -0.004116841485984027]`
- **Why:** A new episode: a list of four small random floats.

```python
s = discretize(state)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.046035290461553954, 0.02189762200322956, -0.023633383322497127, -0.004116841485984027]`
- **Now `s` holds** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 2, 2)`
- **Why:** The state's table address: a tuple of four ints.

```python
steps = 0
```
- **Now `steps` holds** — type: **int (whole number)**; value: `0`
- **Why:** Counts ticks in this episode.

```python
while True:
```
- **Why:** An inner loop with no stopping condition of its own: it runs until something executes `break` (at failure or 500 ticks). Each pass is one tick.

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 2, 2)`
3. `epsilon` → *float (decimal number)*: `1.0`
- **Now `a` holds** — type: **int (whole number)**; value: `0`
- **Why:** Picks 0 or 1 by the epsilon-greedy rule from Step 16. At epsilon 1.0 it is always random.

```python
state = step(state, a)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.046035290461553954, 0.02189762200322956, -0.023633383322497127, -0.004116841485984027]`
2. `a` → *int (whole number)*: `0`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[0.04647324290161855, -0.17287755979987587, -0.023715720152216808, 0.28101677527587066]`
- **Why:** The physics moves forward one tick: `state` becomes a new list of four floats.

```python
steps += 1
```
- **Now `steps` holds** — type: **int (whole number)**; value: `1`
- **Why:** One more tick survived.

```python
failed = is_done(state)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.04647324290161855, -0.17287755979987587, -0.023715720152216808, 0.28101677527587066]`
- **Now `failed` holds** — type: **bool (True/False)**; value: `False`
- **Why:** A `bool`: did that tick end the episode?

```python
s_next = discretize(state)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.04647324290161855, -0.17287755979987587, -0.023715720152216808, 0.28101677527587066]`
- **Now `s_next` holds** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 2, 3)`
- **Why:** The address of the situation we arrived in.

```python
reward = 1
```
- **Now `reward` holds** — type: **int (whole number)**; value: `1`
- **Why:** +1 for surviving this tick (even the tick that fails; that is the convention in the standard CartPole).


#### …the same call, continued: `train(1)`

```python
best_next = 0.0 if failed else Q[s_next].max()   # no future after failing
```
Evaluated inside-out (the main pieces):
1. `failed` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
3. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
4. `Q[s_next]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
- **Now `best_next` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** The value of where we landed, **except** after failing: the episode is over, so there is no future, so it is 0. This is what teaches the agent that failing is bad: a failing move's target is only +1, a surviving move's target is +1 plus something positive.

```python
target = reward + gamma * best_next
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `1`
2. `gamma` → *float (decimal number)*: `0.99`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `target` holds** — type: **float (decimal number)**; value: `1.0`
- **Why:** The same target as in Step 17.

```python
Q[s + (a,)] += alpha * (target - Q[s + (a,)])
```
Evaluated inside-out (the main pieces):
1. `s` → *tuple of 4 items (each an int)*: `(1, 1, 2, 2)`
2. `a` → *int (whole number)*: `0`
3. `(a,)` → *tuple of 1 item (each an int)*: `(0,)`
4. `s + (a,)` → *tuple of 5 items (each an int)*: `(1, 1, 2, 2, 0)`
5. `alpha` → *float (decimal number)*: `0.2`
6. `target` → *float (decimal number)*: `1.0`
7. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers, all zero`
8. `Q[s + (a,)]` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers; 1 are non-zero; smallest 0, largest 0.2`
- **Why:** **The learning rule**, in one line (`+=` adds to the existing value). `Q[s + (a,)]` is the single box for (situation, action). `target - Q[...]` is the error and `alpha` the fraction we correct. Exactly one box of 648 changes.

```python
s = s_next
```
- **Now `s` holds** — type: **tuple of 4 items (each an int)**; value: `(1, 1, 2, 3)`
- **Why:** Moves on: the situation we landed in becomes the situation we are in.

```python
if failed or steps >= MAX_STEPS:
```
Evaluated inside-out (the main pieces):
1. `failed` → *bool (True/False)*: `False`
2. `steps` → *int (whole number)*: `1`
3. `MAX_STEPS` → *int (whole number)*: `500`
4. `steps >= MAX_STEPS` → *bool (True/False)*: `False`
5. `failed or steps >= MAX_STEPS` → *bool (True/False)*: `False`
- **Why:** `failed` is a `bool` and `steps >= MAX_STEPS` is another; `or` is true if either is. If true, `break` ends the tick loop and the episode is over.

```python
while True:
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 1 are non-zero; smallest 0, largest 0.2`
2. `s` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
3. `epsilon` → *float (decimal number)*: `1.0`
4. `choose_action(Q, s, epsilon)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
state = step(state, a)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.04647324290161855, -0.17287755979987587, -0.023715720152216808, 0.28101677527587066]`
2. `a` → *int (whole number)*: `0`
- **Now `state` holds** — type: **list of 4 items (each a float)**; value: `[0.04301569170562103, -0.3676533331298385, -0.018095384646699395, 0.5661265093780321]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
steps += 1
```
- **Now `steps` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
failed = is_done(state)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.04301569170562103, -0.3676533331298385, -0.018095384646699395, 0.5661265093780321]`
2. `is_done(state)` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `train(1)`

```python
s_next = discretize(state)
```
Evaluated inside-out (the main pieces):
1. `state` → *list of 4 items (each a float)*: `[0.04301569170562103, -0.3676533331298385, -0.018095384646699395, 0.5661265093780321]`
2. `discretize(state)` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
reward = 1
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if failed else Q[s_next].max()   # no future after failing
```
Evaluated inside-out (the main pieces):
1. `failed` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 1 are non-zero; smallest 0, largest 0.2`
3. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
4. `Q[s_next]` → *numpy array, shape (2,), holding decimal numbers*: `[0., 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
6. `0.0 if failed else Q[s_next].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
target = reward + gamma * best_next
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `1`
2. `gamma` → *float (decimal number)*: `0.99`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
5. `reward + gamma * best_next` → *float (decimal number)*: `1.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s + (a,)] += alpha * (target - Q[s + (a,)])
```
Evaluated inside-out (the main pieces):
1. `s` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
2. `a` → *int (whole number)*: `0`
3. `(a,)` → *tuple of 1 item (each an int)*: `(0,)`
4. `s + (a,)` → *tuple of 5 items (each an int)*: `(1, 1, 2, 3, 0)`
5. `alpha` → *float (decimal number)*: `0.2`
6. `target` → *float (decimal number)*: `1.0`
7. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 1 are non-zero; smallest 0, largest 0.2`
8. `Q[s + (a,)]` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers; 2 are non-zero; smallest 0, largest 0.2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = s_next
```
Evaluated inside-out (the main pieces):
1. `s_next` → *tuple of 4 items (each an int)*: `(1, 1, 2, 3)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 18.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Makes the random start states and exploring moves repeatable.

In [ ]:
# Step 18.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.3: type this

```python
Q, history = train(300)
```

- **Creates `Q`** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers; 110 are non-zero; smallest 0, largest 31.52`

- **Creates `history`** — type: **list of 300 items (each an int)**; value: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...`

- **Why / what it means:** Unpacks the returned tuple into two names: the learned table (an array) and the list of 300 episode lengths. 300 episodes is far too few to learn much; the point is to see the first steps.

In [ ]:
# Step 18.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.4: type this

```python
print(history[:10])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `history` → *list of 300 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...`
2. `history[:10]` → *list of 10 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36]`

**Prints:**
```text
[16, 16, 21, 11, 15, 24, 21, 25, 36, 36]
```

In [ ]:
# Step 18.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.5: type this

```python
print(sum(history[:100]) / 100, sum(history[200:300]) / 100)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `history` → *list of 300 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...`
2. `history[:100]` → *list of 100 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...`
3. `sum(history[:100])` → *int (whole number)*: `2362`
4. `sum(history[:100]) / 100` → *float (decimal number)*: `23.62`
5. `history[200:300]` → *list of 100 items (each an int)*: `[31, 26, 12, 20, 22, 17, 19, 33, 16, 21, 38, 29, 15, 42, 14, 58, 39, 14, 11, 11, 48, 49, 17, 58, 57, 13, 47, 38, 26, 25, 21, 31...`
6. `sum(history[200:300])` → *int (whole number)*: `2618`
7. `sum(history[200:300]) / 100` → *float (decimal number)*: `26.18`

**Prints:**
```text
23.62 26.18
```

In [ ]:
# Step 18.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18 complete: everything it printed, and what it means

```text
[16, 16, 21, 11, 15, 24, 21, 25, 36, 36]
23.62 26.18
```

Three hundred episodes is far too few to see much, which is the point of showing it: the first ten episodes last 11 to 36 ticks (basically random), and the average moves only from `23.62` (episodes 0 to 99) to `26.18` (episodes 200 to 299). Learning has barely started. Real progress needs thousands of episodes.

## Step 19: The real training run, and a fair test

**The idea first.** We train for 4000 episodes, summarise progress in blocks, and then test the finished table with **no exploring at all**: the agent just takes the best action every time.

This step has **9 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 19.1: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness.

In [ ]:
# Step 19.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.2: type this

```python
Q, history = train(4000)
```

- **Creates `Q`** — type: **numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers**; value: `648 numbers; 368 are non-zero; smallest 0, largest 89.2`

- **Creates `history`** — type: **list of 4000 items (each an int)**; value: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...`

- **Why / what it means:** The real training run: 4000 episodes. It takes about ten seconds because the loop body runs hundreds of thousands of times.

In [ ]:
# Step 19.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.3: type this

```python
for i in range(0, 4000, 500):
    print("episodes", i, "to", i + 499, ": average", round(sum(history[i:i + 500]) / 500, 1))
```

- **Creates `i`** — type: **int (whole number)**; value: `3500`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `range(0, 4000, 500)` → *range*: `range(0, 4000, 500)`
2. `i` → *int (whole number)*: `0` *(evaluated 8 times; the first is shown)*
3. `i + 499` → *int (whole number)*: `499` *(evaluated 8 times; the first is shown)*
4. `history` → *list of 4000 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...` *(evaluated 8 times; the first is shown)*
5. `history[i:i + 500]` → *list of 500 items (each an int)*: `[16, 16, 21, 11, 15, 24, 21, 25, 36, 36, 16, 31, 39, 13, 25, 36, 34, 14, 10, 86, 11, 26, 11, 9, 23, 11, 19, 17, 16, 23, 16, 29,...` *(evaluated 8 times; the first is shown)*
6. `sum(history[i:i + 500])` → *int (whole number)*: `14561` *(evaluated 8 times; the first is shown)*
7. `sum(history[i:i + 500]) / 500` → *float (decimal number)*: `29.122` *(evaluated 8 times; the first is shown)*
8. `round(sum(history[i:i + 500]) / 500, 1)` → *float (decimal number)*: `29.1` *(evaluated 8 times; the first is shown)*

**Prints:**
```text
episodes 0 to 499 : average 29.1
episodes 500 to 999 : average 53.4
episodes 1000 to 1499 : average 77.3
episodes 1500 to 1999 : average 92.5
episodes 2000 to 2499 : average 119.1
episodes 2500 to 2999 : average 187.6
episodes 3000 to 3499 : average 248.6
episodes 3500 to 3999 : average 334.6
```

- **Why / what it means:** `range(0, 4000, 500)` gives 0, 500, 1000, ... 3500 (the third number is the step size). For each `i`, `history[i:i + 500]` is a slice of 500 episode lengths, and `sum(...) / 500` averages them; `round(x, 1)` keeps one decimal place.

In [ ]:
# Step 19.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.4: type this

```python
print("boxes ever changed:", np.count_nonzero(Q), "of", Q.size)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (3, 3, 6, 6, 2), holding decimal numbers*: `648 numbers; 368 are non-zero; smallest 0, largest 89.2`
2. `np.count_nonzero(Q)` → *numpy integer (a whole number)*: `368`
3. `Q.size` → *int (whole number)*: `648`

**Prints:**
```text
boxes ever changed: 368 of 648
```

In [ ]:
# Step 19.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.5: type this

```python
def greedy_policy(state):
    return int(np.argmax(Q[discretize(state)]))
```

- **Creates** `greedy_policy` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A policy in the same shape as the others (state in, action out), but built from the table: discretize the state, look up its row, take the position of the biggest number. No randomness at all.

In [ ]:
# Step 19.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.6: type this

```python
random.seed(5)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(5)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable test starts.

In [ ]:
# Step 19.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.7: type this

```python
scores = [run_episode(greedy_policy) for _ in range(20)]
```

- **Creates `scores`** — type: **list of 20 items (each an int)**; value: `[275, 170, 136, 274, 277, 265, 168, 280, 269, 214, 319, 345, 350, 124, 318, 300, 158, 163, 282, 160]`

- **Why / what it means:** Twenty test episodes using only the table's best actions, collected into a list of 20 ints. This is the fair test of what was learned.

In [ ]:
# Step 19.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.8: type this

```python
print(scores)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *list of 20 items (each an int)*: `[275, 170, 136, 274, 277, 265, 168, 280, 269, 214, 319, 345, 350, 124, 318, 300, 158, 163, 282, 160]`

**Prints:**
```text
[275, 170, 136, 274, 277, 265, 168, 280, 269, 214, 319, 345, 350, 124, 318, 300, 158, 163, 282, 160]
```

In [ ]:
# Step 19.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.9: type this

```python
print(sum(scores) / len(scores))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `scores` → *list of 20 items (each an int)*: `[275, 170, 136, 274, 277, 265, 168, 280, 269, 214, 319, 345, 350, 124, 318, 300, 158, 163, 282, 160]`
2. `sum(scores)` → *int (whole number)*: `4847`
3. `len(scores)` → *int (whole number)*: `20`
4. `sum(scores) / len(scores)` → *float (decimal number)*: `242.35`

**Prints:**
```text
242.35
```

In [ ]:
# Step 19.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19 complete: everything it printed, and what it means

```text
episodes 0 to 499 : average 29.1
episodes 500 to 999 : average 53.4
episodes 1000 to 1499 : average 77.3
episodes 1500 to 1999 : average 92.5
episodes 2000 to 2499 : average 119.1
episodes 2500 to 2999 : average 187.6
episodes 3000 to 3499 : average 248.6
episodes 3500 to 3999 : average 334.6
boxes ever changed: 368 of 648
[275, 170, 136, 274, 277, 265, 168, 280, 269, 214, 319, 345, 350, 124, 318, 300, 158, 163, 282, 160]
242.35
```

This takes about 10 seconds. The average episode length climbs from **29** (random level) to **335**, an eleven-fold improvement, using only reward feedback and 648 numbers. Nobody told the agent the physics or the rule from Step 12. Only **368 of 648 boxes** were ever changed: the others are situations that a decent policy never reaches, such as the pole spinning wildly while the cart is far off the track. The greedy test averages **242**, about eight times random, and about equal to the hand-written rule. Individual test episodes range from 124 to 350, because starts differ and 324 buckets are a coarse picture of the real state. Note that the training averages include random exploring moves, so they understate the table.

## Step 20: A learning curve you can read

**The idea first.** Numbers in blocks are hard to see. A bar for each stretch of episodes makes the trend visible, without needing a plotting library.

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 20.1: type this

```python
averages = [sum(history[i:i + 100]) / 100 for i in range(0, 4000, 100)]
```

- **Creates `averages`** — type: **list of 40 items (each a float)**; value: `[23.62, 24.17, 26.18, 34.43, 37.21, 44.19, 44.92, 59.18, 60.0, 58.8, 81.97, 68.78, 78.21, 75.43, 82.06, 89.54, 89.42, 88.67, 93...`

- **Why / what it means:** `history[i:i + 100]` is a **slice**: a sub-list of 100 consecutive items starting at position `i`. `sum(...) / 100` averages them. `range(0, 4000, 100)` produces 0, 100, 200, ... 3900 (the third number is the step size). The result is a list of 40 floats: the average episode length in each block of 100 episodes.

In [ ]:
# Step 20.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.2: type this

```python
for i, avg in enumerate(averages):
    if i % 2 == 0:
        print(f"{i * 100:5d} {'#' * int(avg / 6)} {avg:.0f}")
```

- **Creates `i`** — type: **int (whole number)**; value: `39`

- **Creates `avg`** — type: **float (decimal number)**; value: `266.75`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `averages` → *list of 40 items (each a float)*: `[23.62, 24.17, 26.18, 34.43, 37.21, 44.19, 44.92, 59.18, 60.0, 58.8, 81.97, 68.78, 78.21, 75.43, 82.06, 89.54, 89.42, 88.67, 93...`
2. `enumerate(averages)` → *enumerate*: `<enumerate object at 0x7fa1833d3bf0>`
3. `i` → *int (whole number)*: `0` *(evaluated 40 times; the first is shown)*
4. `i % 2` → *int (whole number)*: `0` *(evaluated 40 times; the first is shown)*
5. `i % 2 == 0` → *bool (True/False)*: `True` *(evaluated 40 times; the first is shown)*
6. `i * 100` → *int (whole number)*: `0` *(evaluated 20 times; the first is shown)*
7. `avg` → *float (decimal number)*: `23.62` *(evaluated 20 times; the first is shown)*
8. `avg / 6` → *float (decimal number)*: `3.936666666666667` *(evaluated 20 times; the first is shown)*
9. `int(avg / 6)` → *int (whole number)*: `3` *(evaluated 20 times; the first is shown)*
10. `'#' * int(avg / 6)` → *str (text, 3 characters)*: `'###'` *(evaluated 20 times; the first is shown)*

**Prints:**
```text
    0 ### 24
  200 #### 26
  400 ###### 37
  600 ####### 45
  800 ########## 60
 1000 ############# 82
 1200 ############# 78
 1400 ############# 82
 1600 ############## 89
 1800 ############### 94
 2000 ################# 107
 2200 ################## 110
 2400 ################### 114
 2600 ################################## 206
 2800 ############################# 176
 3000 ########################
...
```

- **Why / what it means:** `enumerate` pairs each item with its position: `(0, 24.1)`, `(1, ...)` and so on, unpacked into `i` and `avg`. `if i % 2 == 0` keeps every second block (`%` is the remainder after dividing, so it is 0 for even numbers). `'#' * int(avg / 6)` repeats the character `#` that many times, so a larger average draws a longer bar: strings can be multiplied just like lists.

In [ ]:
# Step 20.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20 complete: everything it printed, and what it means

```text
    0 ### 24
  200 #### 26
  400 ###### 37
  600 ####### 45
  800 ########## 60
 1000 ############# 82
 1200 ############# 78
 1400 ############# 82
 1600 ############## 89
 1800 ############### 94
 2000 ################# 107
 2200 ################## 110
 2400 ################### 114
 2600 ################################## 206
 2800 ############################# 176
 3000 ############################## 185
 3200 ################################################### 312
 3400 ############################################### 283
 3600 ############################################## 282
 3800 ######################################################### 344
```

The bars grow from about 24 at the left to over 300 at the right, but not smoothly: look at 2600 (206) followed by 2800 (176), and 3200 (312) followed by 3400 (283). Learning with coarse buckets is **noisy and not monotonic**, so never judge an agent from a single block.

**Try this yourself.** If you have matplotlib, replace the printing with `import matplotlib.pyplot as plt`, then `plt.plot(averages)` and `plt.show()`.

---
# Part D: experiments

You now own a learner. The best way to understand why each part matters is to break it. **Change exactly one thing, write down your prediction, then run.** Each run prints two numbers: the average episode length *while learning* (includes random exploring moves, so it understates the table), and the *greedy test* (20 fresh episodes using only the table's best action, no exploring; a small sample, so noisy).

## Step 21: A helper to run experiments

**The idea first.** We wrap training and testing in one function so each experiment is a single line. It also teaches two useful Python tricks.

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 21.1: type this

```python
def experiment(label, bins=(3, 3, 6, 6), episodes=4000, seed=0, **settings):
    global BINS
    saved_bins = BINS
    BINS = np.array(bins)
    try:
        random.seed(seed)
        Q_local, hist = train(episodes, **settings)
        blocks = [round(sum(hist[i:i + 1000]) / 1000) for i in range(0, episodes, 1000)]

        def policy(state):
            return int(np.argmax(Q_local[discretize(state)]))

        random.seed(99)
        test = [run_episode(policy) for _ in range(20)]
    finally:
        BINS = saved_bins
    print(f"{label:24s} training avg per 1000 episodes {blocks}   greedy test avg {round(sum(test) / 20)}")
```

- **Creates** `experiment` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A wrapper that trains one setting and prints one row. `**settings` collects any extra *named* arguments (like `alpha=0.02`) into a dictionary, and later `train(episodes, **settings)` spreads them back out as named arguments, so the helper does not need to know what they are.

In [ ]:
# Step 21.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.2: type this

```python
experiment("baseline")
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('baseline')` → *None (nothing)*: `None`

**Prints:**
```text
baseline                 training avg per 1000 episodes [41, 85, 153, 292]   greedy test avg 240
```

- **Why / what it means:** Calls the helper with no overrides, so everything takes its default value. The function runs the training and prints one row.

In [ ]:
# Step 21.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21 complete: everything it printed, and what it means

```text
baseline                 training avg per 1000 episodes [41, 85, 153, 292]   greedy test avg 240
```

The baseline reproduces Step 19: training averages of 41, 85, 153, 292 for each block of 1000 episodes, and a greedy test of 240. Every later experiment is compared with these two numbers.

## Step 22: Experiment 1: exploration

**The idea first.** **Predict first:** what happens if the agent never explores (`epsilon = 0`)? And what if it keeps exploring 30% of the time forever (`eps_min=0.3`)?

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 22.1: type this

```python
experiment("no exploring", eps_start=0.0, eps_min=0.0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('no exploring', eps_start=0.0, eps_min=0.0)` → *None (nothing)*: `None`

**Prints:**
```text
no exploring             training avg per 1000 episodes [12, 11, 12, 12]   greedy test avg 13
```

- **Why / what it means:** `eps_start=0.0` makes epsilon start at zero, and `eps_min=0.0` keeps it there.

In [ ]:
# Step 22.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22.2: type this

```python
experiment("eps_min=0.3", eps_min=0.3)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('eps_min=0.3', eps_min=0.3)` → *None (nothing)*: `None`

**Prints:**
```text
eps_min=0.3              training avg per 1000 episodes [41, 83, 93, 100]   greedy test avg 196
```

- **Why / what it means:** Epsilon can decay no lower than 0.3, so 30% of moves stay random forever.

In [ ]:
# Step 22.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22 complete: everything it printed, and what it means

```text
no exploring             training avg per 1000 episodes [12, 11, 12, 12]   greedy test avg 13
eps_min=0.3              training avg per 1000 episodes [41, 83, 93, 100]   greedy test avg 196
```

**No exploring: 12 ticks, worse than random.** Every box starts at 0. In an unseen situation both tie, so a random action is picked. Whichever it is, the update gives that box a *positive* number (the target is at least 1, since the reward is +1). The untried action's box is still 0, so from then on `argmax` always prefers the tried action. The agent locks onto whatever it tried first, in every situation, and can never discover it was wrong. **Constant 30% exploring:** the training average is stuck near 100 while the greedy test is 196, close to the baseline's 240. The table learned fine, but the training episodes were handicapped by constant random pushes. **A training score measures learning plus exploring; the greedy test measures only what was learned.** Always check the greedy test before deciding something made things worse (and with only 20 test episodes, 196 versus 240 is within the noise).

## Step 23: Experiment 2: the learning rate alpha

**The idea first.** **Predict first:** `alpha = 0.02` (ten times smaller) and `alpha = 1.0` (replace the old value completely). Which learns more slowly? Which is unstable?

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 23.1: type this

```python
experiment("alpha=0.02", alpha=0.02)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('alpha=0.02', alpha=0.02)` → *None (nothing)*: `None`

**Prints:**
```text
alpha=0.02               training avg per 1000 episodes [59, 148, 150, 150]   greedy test avg 147
```

- **Why / what it means:** A learning rate ten times smaller than the default.

In [ ]:
# Step 23.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.2: type this

```python
experiment("alpha=1.0", alpha=1.0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('alpha=1.0', alpha=1.0)` → *None (nothing)*: `None`

**Prints:**
```text
alpha=1.0                training avg per 1000 episodes [22, 29, 34, 42]   greedy test avg 96
```

- **Why / what it means:** A learning rate of 1: each update replaces the old value completely.

In [ ]:
# Step 23.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23 complete: everything it printed, and what it means

```text
alpha=0.02               training avg per 1000 episodes [59, 148, 150, 150]   greedy test avg 147
alpha=1.0                training avg per 1000 episodes [22, 29, 34, 42]   greedy test avg 96
```

**`alpha = 0.02`** surprised me. I expected it to be slower, and at first it was *faster* (59 and 148 in the first two blocks against 41 and 85) before flattening at 150 while the baseline kept climbing to 292. A guess for why (a guess, not proven): small steps smooth out the noise from each coarse bucket mixing different real situations, but they also spread the "good news" about long-term survival too slowly for 4000 episodes. **`alpha = 1.0`** never learns (training stays at 22 to 42). With `alpha = 1` the update simplifies to `new = target`: the old value is thrown away every time. Because several different real situations share a bucket, the same box keeps receiving different targets and swings to whichever came last, never averaging them. Alpha trades speed against stability. When we do the maze later, alpha = 1 will work, and understanding *why the maze differs* is a good test of your understanding.

## Step 24: Experiment 3: the discount factor gamma

**The idea first.** **Predict first:** gamma = 0.5 and gamma = 0.9. Use Step 13's numbers: how much is a reward 10 ticks ahead worth? (`0.5 ** 10` is about 0.001; `0.9 ** 10` is about 0.35; `0.99 ** 10` is about 0.90.)

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 24.1: type this

```python
experiment("gamma=0.5", gamma=0.5)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('gamma=0.5', gamma=0.5)` → *None (nothing)*: `None`

**Prints:**
```text
gamma=0.5                training avg per 1000 episodes [37, 75, 97, 112]   greedy test avg 144
```

- **Why / what it means:** A short horizon: rewards two ticks away are worth only a quarter.

In [ ]:
# Step 24.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 24.2: type this

```python
experiment("gamma=0.9", gamma=0.9)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('gamma=0.9', gamma=0.9)` → *None (nothing)*: `None`

**Prints:**
```text
gamma=0.9                training avg per 1000 episodes [38, 84, 179, 235]   greedy test avg 227
```

- **Why / what it means:** A medium horizon (about ten ticks).

In [ ]:
# Step 24.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 24 complete: everything it printed, and what it means

```text
gamma=0.5                training avg per 1000 episodes [37, 75, 97, 112]   greedy test avg 144
gamma=0.9                training avg per 1000 episodes [38, 84, 179, 235]   greedy test avg 227
```

**`gamma = 0.5`** learns but stalls (112 in the last block, test 144): a failure 10 ticks away is worth almost nothing to this agent, so it cannot connect a bad push to the fall it causes ten ticks later. **`gamma = 0.9`** reaches 235 and a test of 227, close to the baseline's 292 and 240. The gap between 0.5 and 0.9 is much bigger than the gap between 0.9 and 0.99. The horizon only has to be long enough to see how failures develop.

## Step 25: Experiment 4: what the agent is allowed to see

**The idea first.** **Predict first:** `bins=(1, 1, 1, 1)` is one bucket per number, so the table has a single row. `bins=(1, 1, 6, 1)` shows only the angle. `bins=(1, 1, 6, 12)` ignores position and velocity but sees the spin finely. Which is best?

This step has **3 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 25.1: type this

```python
experiment("one bucket", bins=(1, 1, 1, 1))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('one bucket', bins=(1, 1, 1, 1))` → *None (nothing)*: `None`

**Prints:**
```text
one bucket               training avg per 1000 episodes [16, 11, 10, 9]   greedy test avg 9
```

- **Why / what it means:** `bins=(1, 1, 1, 1)`: a **tuple** of four ints giving the bucket count for each state number. All ones means every state falls in the same bucket.

In [ ]:
# Step 25.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 25.2: type this

```python
experiment("angle only", bins=(1, 1, 6, 1))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('angle only', bins=(1, 1, 6, 1))` → *None (nothing)*: `None`

**Prints:**
```text
angle only               training avg per 1000 episodes [22, 25, 29, 34]   greedy test avg 23
```

- **Why / what it means:** Six buckets for the angle, one for everything else.

In [ ]:
# Step 25.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 25.3: type this

```python
experiment("angle + spin", bins=(1, 1, 6, 12))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('angle + spin', bins=(1, 1, 6, 12))` → *None (nothing)*: `None`

**Prints:**
```text
angle + spin             training avg per 1000 episodes [65, 122, 146, 169]   greedy test avg 500
```

- **Why / what it means:** Six buckets for the angle and twelve for the spin; position and velocity are ignored.

In [ ]:
# Step 25.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 25 complete: everything it printed, and what it means

```text
one bucket               training avg per 1000 episodes [16, 11, 10, 9]   greedy test avg 9
angle only               training avg per 1000 episodes [22, 25, 29, 34]   greedy test avg 23
angle + spin             training avg per 1000 episodes [65, 122, 146, 169]   greedy test avg 500
```

**One bucket: 9 ticks.** With one row every situation looks identical, so only a single fixed behaviour is possible. **Angle only: 23** (barely better than random). A pole at 0.05 rad that is swinging away needs a different push from one swinging back, and without the spin number the two look the same. **Angle plus spin: a perfect greedy test of 500**, with only 72 situations. Fewer rows means each is visited far more often, so the table fills in faster (65 and 122 in the first two blocks, against the baseline's 41 and 85). Note again that the training average (169) understates a perfect policy because of exploration. A table can only be as smart as the distinctions its rows allow, and more detail is not automatically better.

---
# Part E: beyond tables

The table worked, but look at what it cost: 648 numbers, thousands of episodes, and a clumsy chopping of the state into buckets. Is there a smarter way to represent "what to do"? Let us find out.

## Step 26: A different idea: a policy that is just four numbers

**The idea first.** So far the agent learned *values* in a table and derived a policy from them. A policy could instead be a **formula** with a few adjustable numbers. Here is the simplest: multiply each of the four state numbers by a weight, add the results, and push right if the total is positive. Four weights, no table, no buckets.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 26.1: type this

```python
weights = np.array([0.5, 1.0, 2.0, 1.0])  # one weight per state number
```

- **Creates `weights`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.5, 1., 2., 1.]`

- **Why / what it means:** An array of four floats: one **weight** per state number. A weight says how much that state number influences the decision.

In [ ]:
# Step 26.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26.2: type this

```python
state = np.array([0.0, 0.0, 0.05, 0.0])
```

- **Creates `state`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0.05, 0.]`

- **Why / what it means:** A state as an array (rather than a list), so it can be multiplied by the weights.

In [ ]:
# Step 26.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26.3: type this

```python
print(np.dot(weights, state))             # dot product: multiply pairwise, then add up
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `weights` → *numpy array, shape (4,), holding decimal numbers*: `[0.5, 1., 2., 1.]`
2. `state` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0.05, 0.]`
3. `np.dot(weights, state)` → *float (decimal number)*: `0.1`

**Prints:**
```text
0.1
```

In [ ]:
# Step 26.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26.4: type this

```python
print(0.5*0.0 + 1.0*0.0 + 2.0*0.05 + 1.0*0.0)   # the same thing written out
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `0.5 * 0.0` → *float (decimal number)*: `0.0`
2. `1.0 * 0.0` → *float (decimal number)*: `0.0`
3. `0.5 * 0.0 + 1.0 * 0.0` → *float (decimal number)*: `0.0`
4. `2.0 * 0.05` → *float (decimal number)*: `0.1`
5. `0.5 * 0.0 + 1.0 * 0.0 + 2.0 * 0.05` → *float (decimal number)*: `0.1`
6. `0.5 * 0.0 + 1.0 * 0.0 + 2.0 * 0.05 + 1.0 * 0.0` → *float (decimal number)*: `0.1`

**Prints:**
```text
0.1
```

In [ ]:
# Step 26.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26.5: type this

```python
def make_linear_policy(weights):
    def policy(state):
        return 1 if np.dot(weights, state) > 0 else 0     # push right if the score is positive
    return policy
```

- **Creates** `make_linear_policy` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A function that **returns a function**. Calling `make_linear_policy(weights)` builds and returns the inner function `policy`, which remembers the `weights` it was made with. This pattern is called a *closure*. It lets us create many different policies, one per set of weights.

In [ ]:
# Step 26.5: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `make_linear_policy(np.array([0.0, 0.0, 1.0, 1.0]))`

**What the function receives** (its inputs, called *parameters*):
- `weights` — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 1., 1.]`

**Now each line runs, top to bottom:**

```python
def policy(state):
    return 1 if np.dot(weights, state) > 0 else 0     # push right if the score is positive
```
- **Now `policy` holds** — type: **function (a reusable recipe)**; value: `<function policy>`
- **Why:** Defines the inner function. It is created fresh each time `make_linear_policy` runs, so each policy keeps its own weights.

```python
return policy
```
- **Why:** The maker hands back the inner function itself (a **function**, not a number). The caller can now call it like any other policy: `policy(state)`.

**The function hands back** (its *return value*) — type: **function (a reusable recipe)**; value: `<function policy>`


### Step 26.6: type this

```python
policy = make_linear_policy(np.array([0.0, 0.0, 1.0, 1.0]))
```

- **Creates `policy`** — type: **function (a reusable recipe)**; value: `<function policy>`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.array([0.0, 0.0, 1.0, 1.0])` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 1., 1.]`

- **Why / what it means:** Calls the maker with weights `[0, 0, 1, 1]`. The value `policy` is now a **function**, not a number: calling it will compute `1 if np.dot(weights, state) > 0 else 0`.

In [ ]:
# Step 26.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26.7: type this

```python
print(policy([0.0, 0.0, 0.05, 0.0]), policy([0.0, 0.0, -0.05, 0.0]))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `policy([0.0, 0.0, 0.05, 0.0])` → *int (whole number)*: `1`
2. `[0.0, 0.0, -0.05, 0.0]` → *list of 4 items (each a float)*: `[0.0, 0.0, -0.05, 0.0]`
3. `policy([0.0, 0.0, -0.05, 0.0])` → *int (whole number)*: `0`

**Prints:**
```text
1 0
```

In [ ]:
# Step 26.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 26 complete: everything it printed, and what it means

```text
0.1
0.1
1 0
```

`0.1` and `0.1`: the dot product equals the long-hand sum (only the angle contributes: `2.0 x 0.05`). The policy pushes right (`1`) when the pole leans right and left (`0`) when it leans left, which is a sensible instinct. Why might a *linear* rule work? Near upright, the physics is almost a straight-line relationship between the state and the right push, so a weighted sum can capture it.

## Step 27: Finding good weights by random search

**The idea first.** How do we find good weights? The crudest method possible: try random ones, and keep the best. This is **random search**. It does not use the update rule at all.

This step has **9 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 27.1: type this

```python
def average_score(policy, episodes=5):
    return sum(run_episode(policy) for _ in range(episodes)) / episodes
```

- **Creates** `average_score` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Averages the score over several episodes. One episode alone is too noisy to judge a policy by.

In [ ]:
# Step 27.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness for Python's `random` (used for the episode starts).

In [ ]:
# Step 27.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.3: type this

```python
np.random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** A *separate* generator: numpy has its own, used here for the random weights, so it needs its own seed.

In [ ]:
# Step 27.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.4: type this

```python
best_weights = None
best_score = -1
```

- **Creates `best_weights`** — type: **None (nothing)**; value: `None`

- **Why / what it means:** `None` means 'nothing yet'. We will replace it when we find something better.

- **Creates `best_score`** — type: **int (whole number)**; value: `-1`

- **Why / what it means:** A score lower than any real score, so the first trial always beats it.

In [ ]:
# Step 27.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.5: type this

```python
for trial in range(1, 301):
    weights = np.random.uniform(-1, 1, 4)              # four random numbers in [-1, 1]
    score = average_score(make_linear_policy(weights))
    if score > best_score:
        best_score = score
        best_weights = weights
        print("trial", trial, "weights", np.round(weights, 2), "score", score)
    if best_score >= MAX_STEPS:
        break
```

- **Creates `trial`** — type: **int (whole number)**; value: `10`

- **Creates `weights`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.2242, 0.2339, 0.8875, 0.3636]`

- **Creates `score`** — type: **float (decimal number)**; value: `500.0`

- **Creates `best_score`** — type: **float (decimal number)**; value: `500.0`

- **Creates `best_weights`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.2242, 0.2339, 0.8875, 0.3636]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `range(1, 301)` → *range*: `range(1, 301)`
2. `np.random.uniform(-1, 1, 4)` → *numpy array, shape (4,), holding decimal numbers*: `[0.0976, 0.4304, 0.2055, 0.0898]` *(evaluated 10 times; the first is shown)*
3. `weights` → *numpy array, shape (4,), holding decimal numbers*: `[0.0976, 0.4304, 0.2055, 0.0898]` *(evaluated 10 times; the first is shown)*
4. `average_score(make_linear_policy(weights))` → *float (decimal number)*: `9.8` *(evaluated 10 times; the first is shown)*
5. `score` → *float (decimal number)*: `9.8` *(evaluated 10 times; the first is shown)*
6. `best_score` → *int (whole number)*: `-1` *(evaluated 10 times; the first is shown)*
7. `score > best_score` → *bool (True/False)*: `True` *(evaluated 10 times; the first is shown)*
8. `trial` → *int (whole number)*: `1` *(evaluated 3 times; the first is shown)*
9. `np.round(weights, 2)` → *numpy array, shape (4,), holding decimal numbers*: `[0.1, 0.43, 0.21, 0.09]` *(evaluated 3 times; the first is shown)*
10. `MAX_STEPS` → *int (whole number)*: `500` *(evaluated 10 times; the first is shown)*
11. `best_score >= MAX_STEPS` → *bool (True/False)*: `False` *(evaluated 10 times; the first is shown)*

**Prints:**
```text
trial 1 weights [0.1  0.43 0.21 0.09] score 9.8
trial 2 weights [-0.15  0.29 -0.12  0.78] score 433.4
trial 10 weights [0.22 0.23 0.89 0.36] score 500.0
```

- **Why / what it means:** The search loop: each trial draws four random weights in [-1, 1], builds a policy from them, and judges it. If it beats the best so far we remember it and print. The loop `break`s as soon as a perfect score is reached.

In [ ]:
# Step 27.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.6: type this

```python
print("trials used:", trial)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `trial` → *int (whole number)*: `10`

**Prints:**
```text
trials used: 10
```

In [ ]:
# Step 27.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.7: type this

```python
final = make_linear_policy(best_weights)
```

- **Creates `final`** — type: **function (a reusable recipe)**; value: `<function policy>`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `best_weights` → *numpy array, shape (4,), holding decimal numbers*: `[0.2242, 0.2339, 0.8875, 0.3636]`

- **Why / what it means:** Rebuilds the best policy for a fresh test, to check it was not just lucky.

In [ ]:
# Step 27.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.8: type this

```python
random.seed(7)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(7)` → *None (nothing)*: `None`

- **Why / what it means:** A different seed from the search, so the test episodes are new.

In [ ]:
# Step 27.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27.9: type this

```python
print([run_episode(final) for _ in range(20)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[run_episode(final) for _ in range(20)]` → *list of 20 items (each an int)*: `[500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]`

**Prints:**
```text
[500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
```

In [ ]:
# Step 27.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 27 complete: everything it printed, and what it means

```text
trial 1 weights [0.1  0.43 0.21 0.09] score 9.8
trial 2 weights [-0.15  0.29 -0.12  0.78] score 433.4
trial 10 weights [0.22 0.23 0.89 0.36] score 500.0
trials used: 10
[500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
```

Trial 1 scores about 10. Trial 2 scores **433**. Trial 10 scores a perfect **500.0**, and the search stops. On 20 fresh test episodes the winning weights survive all 500 ticks every time. So **ten random tries, each judged over five episodes, beat the table's 242**, using four numbers instead of 648 and no learning rule at all. The lesson is not that random search is a great algorithm (it would fail on large problems). It is that **how you represent the solution can matter more than which algorithm you use**. The bucketed table throws away detail and needs thousands of episodes; the linear formula matches the physics and is nearly trivial to find. Modern methods (next steps) mostly try to get the best of both: learn flexibly, from reward, with a smart representation.

**Try this yourself.** Change the 5 in `average_score` to 1 and rerun with different seeds. Does the search ever pick a policy that looks perfect but fails on the 20 fresh episodes? That is what "lucky" means.

## Step 28: Check against the official CartPole

**The idea first.** Everything so far used physics we wrote ourselves. Is it faithful? The standard version used by researchers is **CartPole-v1** in the `gymnasium` library. Running the same policies there is a good check, and it is how you would work on real projects.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 28.1: type this

```python
import gymnasium as gym                  # pip install gymnasium
```

- **Creates `gym`** — type: **module (a library of tools)**; value: `<module gymnasium>`

In [ ]:
# Step 28.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.2: type this

```python
env = gym.make("CartPole-v1")             # the standard, official version
```

- **Creates `env`** — type: **TimeLimit**; value: `<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>`

- **Why / what it means:** Creates the official environment object. It bundles `reset` and `step` like ours, but written by experts and used by researchers worldwide.

In [ ]:
# Step 28.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.3: type this

```python
obs, info = env.reset(seed=0)             # returns (observation, extra info)
```

- **Creates `obs`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0.0137, -0.023, -0.0459, -0.0483]`

- **Creates `info`** — type: **dict with 0 entries**; value: `{}`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `env` → *TimeLimit*: `<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>`

- **Why / what it means:** `reset` returns **two** values; we unpack them into `obs` (the observation: four numbers) and `info` (extra data we ignore).

In [ ]:
# Step 28.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.4: type this

```python
print(obs)                                # 4 numbers: the same four state values as ours
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `obs` → *numpy array, shape (4,), holding decimal numbers*: `[0.0137, -0.023, -0.0459, -0.0483]`

**Prints:**
```text
[ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
```

In [ ]:
# Step 28.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.5: type this

```python
def run_gym(policy, episodes=10):
    scores = []
    for i in range(episodes):
        obs, info = env.reset(seed=i)
        ticks = 0
        while True:
            action = policy(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            ticks += 1
            if terminated or truncated:
                break
        scores.append(ticks)
    return scores
```

- **Creates** `run_gym` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The same idea as `run_episode`, adapted to the library's interface: `env.step` returns five values, so we unpack five names.

In [ ]:
# Step 28.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.6: type this

```python
print(run_gym(lambda o: 1 if o[3] > 0 else 0))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `run_gym(lambda o: 1 if o[3] > 0 else 0)` → *list of 10 items (each an int)*: `[142, 161, 179, 205, 138, 244, 222, 176, 192, 223]`

**Prints:**
```text
[142, 161, 179, 205, 138, 244, 222, 176, 192, 223]
```

In [ ]:
# Step 28.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.7: type this

```python
print(run_gym(lambda o: 1 if o[2] + o[3] > 0 else 0))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `run_gym(lambda o: 1 if o[2] + o[3] > 0 else 0)` → *list of 10 items (each an int)*: `[334, 500, 500, 500, 500, 500, 500, 500, 500, 500]`

**Prints:**
```text
[334, 500, 500, 500, 500, 500, 500, 500, 500, 500]
```

In [ ]:
# Step 28.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28.8: type this

```python
print(run_gym(make_linear_policy(best_weights)))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `best_weights` → *numpy array, shape (4,), holding decimal numbers*: `[0.2242, 0.2339, 0.8875, 0.3636]`
2. `run_gym(make_linear_policy(best_weights))` → *list of 10 items (each an int)*: `[500, 500, 500, 500, 500, 500, 500, 500, 500, 500]`

**Prints:**
```text
[500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
```

In [ ]:
# Step 28.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 28 complete: everything it printed, and what it means

```text
[ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
[142, 161, 179, 205, 138, 244, 222, 176, 192, 223]
[334, 500, 500, 500, 500, 500, 500, 500, 500, 500]
[500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
```

Your numbers may differ a little from mine, because they depend on the library's version and seeding. The pattern should hold: the spin rule scores around 140 to 245 per episode (like ours, about 200), and **our four-number linear policy scores 500 in the official environment too** (mine gave nine 500s in ten and a 334). The simple angle-plus-spin rule does nearly as well. That means our hand-written physics matches the real thing closely, and that what you learned transfers directly to the standard tools.

---
# Where the wider field goes from here

You have now met three ways to solve CartPole, which stand for three big families of methods.

| Method | What it learns | In this notebook |
|---|---|---|
| **Tabular Q-learning** | A table of values; the policy is "best action in the row" | Steps 13 to 25 |
| **Policy search** | The policy itself, as a few adjustable numbers | Step 27 (random search) |
| **Deep Q-learning (DQN)** | The Q function, as a **neural network** that takes the raw four numbers (no buckets) | Not in this notebook. The maze notebook builds one from scratch. |
| **Policy gradient (REINFORCE)** | A policy network, nudged toward actions that led to higher return | A natural next step |
| **Actor-critic, PPO** | Both a policy and a value function together; the workhorses of modern RL | Beyond that |

**Good next reading (free):** *Reinforcement Learning: An Introduction* by Sutton and Barto (the standard textbook, available free online); the Gymnasium documentation; and OpenAI's "Spinning Up in Deep RL" guide.

**A project idea:** replace the bucketing with a neural network that takes the four raw numbers and outputs two Q values. The maze notebook (Part E) writes exactly this machinery with a one-hot input; for CartPole you change only the input size to 4 and the output size to 2.

# Cheat sheet: the numpy and Python you used

| You wrote | It means |
|---|---|
| `np.array([1, 2, 3])` | make an array from a list |
| `a.shape`, `a.size` | the size in each dimension; the total item count |
| `a[2, 1]` or `a[(2, 1)]` | row 2, column 1 |
| `a[:, 0]`, `a[1]`, `a[0, 1:]` | a column; a row; part of a row |
| `a.reshape(3, 2)` | same items, new shape |
| `np.zeros((3, 4))` | a 3x4 array of zeros |
| `a.max(axis=1)` | best number in each row |
| `a.argmax(axis=1)` | position of the best number in each row |
| `a > 1`, `a == 5` | True/False mask, item by item |
| `np.flatnonzero(mask)` | positions where the mask is True |
| `np.clip(a, lo, hi)` | force every item into a range |
| `a.astype(int)` | chop off decimals |
| `np.dot(w, s)` | multiply pairwise, then add up |
| `random.seed(n)` | make randomness repeatable |
| `f"{x:6.1f}"` | print x as a decimal, 6 wide, 1 decimal |
| `a, b = pair` | unpack a tuple or list |
| `[f(n) for n in range(5)]` | build a list in one line |
| `**settings` | collect extra named inputs / pass them on |

# Glossary of reinforcement learning words

- **Environment:** the world; given a state and an action it returns the next state and a reward.
- **State:** everything needed to predict what happens next (here: four numbers).
- **Action:** a choice the agent makes (here: push left or right).
- **Reward:** the immediate score after an action (here: +1 per tick survived).
- **Episode:** one attempt from a start to a failure or a time limit.
- **Policy:** a rule from state to action. Learning is the search for a good one.
- **Return:** total future reward, with later rewards discounted.
- **Discount factor (gamma):** how much a reward one step later is worth, from 0 to 1.
- **Q(s, a):** the estimated return of taking action `a` in state `s` and then playing well.
- **Learning rate (alpha):** how far each update moves toward its target.
- **Epsilon-greedy:** act randomly with probability epsilon, otherwise take the best known action.
- **Exploration / exploitation:** trying new things versus using what you know.
- **Discretization:** chopping continuous numbers into buckets so a table can hold them.
- **Greedy policy:** always take the action with the highest Q.

# Check yourself

Say each answer out loud before reading it.

1. **What does `Q[s, a]` mean?** The estimated total future reward from taking `a` in `s` and then playing well.
2. **What is the update, in words?** Move the old estimate a fraction `alpha` of the way toward `reward + gamma x (best Q in the row I landed in)`.
3. **Why is `best_next` 0 after failure?** There is no future after failing. This is what makes failing look worse than surviving.
4. **Why does `epsilon = 0` fail in CartPole?** All rewards are positive, so the first action tried in a situation always beats untried ones (which are still 0), and the agent repeats it forever.
5. **Why is a training score misleading?** It includes the random exploring moves. Test with no exploring.
6. **Why did the linear policy beat the table so easily?** Its form matches the physics, so it needs only four numbers; the bucketed table discards detail and must learn 648 separate boxes.
7. **What does discretization cost?** Situations in the same bucket are treated as identical, which blurs the signal and makes learning noisy.